<a href="https://colab.research.google.com/github/adamomariani/oceanshuffle/blob/main/OceanShuffle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

## Environment Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage, set_link_color_palette
from sklearn.metrics import silhouette_score

from google.colab import drive

## Legend

The pipeline is mounted on Google Drive at ['/content/drive/']()

<b>Expects `DB_DIR` pointing to a root directory in Drive with the following structure</b>:
```
.
├── <PANEL_NAME>.parquet        ← required (one of these)
│   <MERGED_DB_NAME>.parquet    ← OR
│   <SOURCE_DB_DIR>/            ← OR (raw .csv files)
├── Output
│   └── Thesis Results
│       └── Figures
│       └── Tables
├── chen2008_annex_volumes.csv  ← optional (for Figure 5.2)

The Output/ directory structure will be created automatically if missing.
Only one of the root-level data files (either <PANEL_NAME> or <MERGED_DB_NAME>), or the <SOURCE_DB_DIR> directory is required.
```


In [ ]:
drive.mount('/content/drive')
# ROOT
DB_DIR = Path("/content/drive/My Drive/oceanshuffle")

# Output/
OUT_DIR = Path(DB_DIR / "Output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Output/Thesis Results
THESIS_RESULTS_DIR = OUT_DIR / "Thesis Results"
THESIS_RESULTS_DIR = Path(THESIS_RESULTS_DIR)
THESIS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
# Output/Thesis Results/Figures
THESIS_FIGURES_DIR = THESIS_RESULTS_DIR / "Figures"
THESIS_FIGURES_DIR = Path(THESIS_FIGURES_DIR)
THESIS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
# Output/Thesis Results/Tables
THESIS_TABLES_DIR = THESIS_RESULTS_DIR / "Tables"
THESIS_TABLES_DIR = Path(THESIS_TABLES_DIR)
THESIS_TABLES_DIR.mkdir(parents=True, exist_ok=True)

# source/          *(only needed for the very first run)
SOURCE_DB_DIR = Path(DB_DIR / "source")

# Main Filenames (either created or to be created)
# - Merged DB will NOT be created if a panel is already present
PANEL_NAME = "hs4_analysis_panel_ch44-49"       # HS4 analysis panel (.parquet extension omitted)
MERGED_DB_NAME = "full_db"                      # Complete merged source database (.parquet extension omitted)

# Main Filepaths (keep intact, change filenames instead)
PANEL_PATH = DB_DIR / f"{PANEL_NAME}.parquet"
MERGED_DB_PATH = DB_DIR / f"{MERGED_DB_NAME}.parquet"

# RUN_PROGRESS:
#   1 = panel already exists at <PANEL_PATH>
#   2 = merged database already exists <MERGED_DB_PATH>
#   3 = none exist yet, source files to merge are in <SOURCE_DB_DIR>
RUN_PROGRESS = 1

# DATABASES
# - OPTION 1: If panel exists, skip DB creation and move on to analysis (default)
# - OPTION 2: If merged database exists, skip merging and move on to panel creation
if not os.path.isfile(PANEL_PATH) and os.path.isfile(MERGED_DB_PATH):
  RUN_PROGRESS = 2
else:
  # - OPTION 3: Check that directory with raw csv files to merge exists as ./source/
  if os.path.isdir(SOURCE_DB_DIR):
    RUN_PROGRESS = 3
  else:
    print(f"SETUP ERROR: Panel could not be created / found")
    print(f"\n- Failed to find panel at [{PANEL_PATH}]")
    print(f"\n- Failed to find merged database to build panel at [{MERGED_DB_PATH}]")
    print(f"\n- Failed to find directory with source files to build merged database in dir [{SOURCE_DB_DIR}]")
    sys.exit(1)

## Analysis Setup

The pipeline uses predefined output folders `THESIS_FIGURES_DIR` and `THESIS_TABLES_DIR`

In [ ]:
STUDY_YEARS = (1995, 2024)   # first and last year of the study period
MIN_FLOW = 10_000            # analysis-sample threshold on reconciled value F, see Section 4.3
FIGURE_FORMATS = ["png"]     # more formats can be added when appropriate (e.g. .pdf)

SAVE_OUTPUTS = True          # set to False to skip writing every table and figure to the drive

# Flagged flows and commodities from Section 4.9 (only exxcluded in the robustness check)
FLAGGED_HS4 = ("4815", "4907")
FLAGGED_FLOWS = "NOT (exporter = 'CAN' AND importer = 'USA' AND hs4 = '4407' AND year IN (2017, 2018))"

HS2_NAMES = {
  "44": "Wood and articles of wood", "45": "Cork", "46": "Plaiting and basketware",
  "47": "Pulp and recovered paper", "48": "Paper and paperboard", "49": "Printed matter",
}

HS4_NAMES = {
  "4401": "Fuel wood & chips", "4402": "Wood charcoal", "4403": "Roundwood (logs)",
  "4404": "Hoopwood, poles, stakes", "4405": "Wood wool & flour", "4406": "Railway sleepers",
  "4407": "Sawnwood", "4408": "Veneer sheets", "4409": "Profiled wood / flooring",
  "4410": "Particle board", "4411": "Fibreboard", "4412": "Plywood", "4413": "Densified wood",
  "4414": "Wooden frames", "4415": "Packing cases & pallets", "4416": "Casks & barrels",
  "4417": "Wooden tools & handles", "4418": "Builders' joinery", "4419": "Wooden tableware",
  "4420": "Marquetry & ornaments", "4421": "Other wood articles",
  "4501": "Natural cork, raw", "4502": "Natural cork, blocks", "4503": "Articles of natural cork",
  "4504": "Agglomerated cork", "4601": "Plaits & plaiting", "4602": "Basketwork & wickerwork",
  "4701": "Mechanical pulp", "4702": "Dissolving pulp", "4703": "Sulphate (kraft) pulp",
  "4704": "Sulphite pulp", "4705": "Semi-chemical pulp", "4706": "Pulp of other fibres",
  "4707": "Recovered paper", "4801": "Newsprint", "4802": "Uncoated printing paper",
  "4803": "Tissue stock", "4804": "Kraft paper & board", "4805": "Other uncoated paper",
  "4806": "Greaseproof & glassine", "4807": "Composite paper", "4808": "Corrugated & creped paper",
  "4809": "Carbon & copying paper", "4810": "Coated paper & board", "4811": "Coated/impregnated paper",
  "4812": "Filter blocks of pulp", "4813": "Cigarette paper", "4814": "Wallpaper",
  "4815": "Paper floor coverings", "4816": "Carbon paper, stencils", "4817": "Envelopes & letter cards",
  "4818": "Toilet & tissue articles", "4819": "Cartons, boxes & bags", "4820": "Registers & notebooks",
  "4821": "Paper labels", "4822": "Bobbins & spools", "4823": "Other paper articles",
  "4901": "Printed books", "4902": "Newspapers & periodicals", "4903": "Children's picture books",
  "4904": "Printed music", "4905": "Maps & charts", "4906": "Plans & drawings",
  "4907": "Stamps, banknotes, cheques", "4908": "Transfers (decals)", "4909": "Postcards & greeting cards",
  "4910": "Calendars", "4911": "Other printed matter",
}

# Colors: use the same color per chapter, per reporting case, and focus/contrast pair throughout
CHAPTER_COLORS = {"44": "#8C510A", "45": "#D9A441", "46": "#7A9A3A", "47": "#3B7EA1", "48": "#1B4F72", "49": "#9B59B6"}
CASE_COLORS = {"both": "#1B4F72", "exporter_only": "#E08E2B", "importer_only": "#5FA8A0", "neither": "#BBBBBB"}
CASE_LABELS = {"both": "Both partners report", "exporter_only": "Exporter only",
               "importer_only": "Importer only", "neither": "Neither (E = I = 0)"}
FOCUS, ACCENT, CONTRAST, GREY = "#1B4F72", "#E08E2B", "#C0392B", "#888888"

### Inflation adjustment

All trade values (`E`, `I`, `F`, and `G`) are converted to constant `PRICE_BASE_YEAR` USD with the annual average US CPI-U (BLS timeseries CUUR0000SA0, for all items, U.S. city average, 1982-84 = 100).
```
Real value = nominal value x CPI[PRICE_BASE_YEAR] / CPI[year].
```
The threshold of the analysis-sample (`MIN_FLOW`) is applied to the *nominal* reconciled value `F_nominal`, since the reconciliation floor used by the HGL dataset andfor this threshold is nominal. Relative `D` and `S` are ratios and are therefore not affected by deflation.

In [ ]:
CPI_U = {
  1995: 152.4, 1996: 156.9, 1997: 160.5, 1998: 163.0, 1999: 166.6, 2000: 172.2, 2001: 177.1,
  2002: 179.9, 2003: 184.0, 2004: 188.9, 2005: 195.3, 2006: 201.6, 2007: 207.342, 2008: 215.303,
  2009: 214.537, 2010: 218.056, 2011: 224.939, 2012: 229.594, 2013: 232.957, 2014: 236.736,
  2015: 237.017, 2016: 240.007, 2017: 245.120, 2018: 251.107, 2019: 255.657, 2020: 258.811,
  2021: 270.970, 2022: 292.655, 2023: 304.702, 2024: 313.689,
}
PRICE_BASE_YEAR = 2024
REAL_USD = True         # Outputs nominal (current USD) results if False

def deflator_table(base_year=PRICE_BASE_YEAR, cpi=CPI_U):
  """Multiplication factor that converts current USD into constant `base_year` USD."""
  return pd.DataFrame({"year": list(cpi), "to_real": [cpi[base_year] / v for v in cpi.values()]})

## Helpers

Set of helper functions and global plotting and labelling styles.

In [ ]:
def set_thesis_style(base_size=9):
  """Matplotlib defaults for all figures."""
  plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": base_size,
    "axes.titlesize": base_size, "axes.labelsize": base_size, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.labelsize": base_size - 1.5, "ytick.labelsize": base_size - 1.5,
    "xtick.direction": "out", "ytick.direction": "out",
    "legend.fontsize": base_size - 1, "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42,
  })


def save_figure(fig, name, formats=FIGURE_FORMATS, directory=None):
  """Save fig as THESIS_FIGURES_DIR/<name>.<fmt> for every format in formats."""
  directory = Path(directory or THESIS_FIGURES_DIR)
  for fmt_ in formats:
    print(f"Saving to: {directory}/{name}.{fmt_}")
    fig.savefig(directory / f"{name}.{fmt_}")


def panel_label(ax, letter, x=-0.12, y=1.04):
  """Bold panel letter (a, b, ...) at the top-left corner"""
  ax.text(x, y, f"({letter})", transform=ax.transAxes, fontweight="bold", va="bottom", ha="left")


def usd(value, decimals=1):
  """Format a USD value with appropriate suffix: k / M / B / T
     e.g. usd(1_234_000_000, 2) => '1.23B', usd(1e4) -> '10k'."""
  for divisor, suffix in [(1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "k"), (1, "")]:
    if abs(value) >= divisor or divisor == 1:
      text = f"{value / divisor:.{decimals}f}"
      if "." in text:
        text = text.rstrip("0").rstrip(".")
      return text + suffix

USD_FORMATTER = FuncFormatter(lambda v, _: usd(v))

def label_points(ax, x, y, labels, fontsize=7, min_gap=0.035, dx=0.012):
  """Write the given labels next to point (x, y), pushing overlapping labels apart vertically.

  min_gap : minimum vertical distance between two labels, as a fraction of the axis height.
  """
  ax.figure.canvas.draw()
  to_axes = ax.transAxes.inverted()
  points = [to_axes.transform(ax.transData.transform((xi, yi))) for xi, yi in zip(x, y)]
  placed = []
  for i in np.argsort([p[1] for p in points]):
    px, py = points[i]
    ty = py
    for qx, qy in sorted(placed, key=lambda p: p[1]):
      if abs(qx - px) < 0.12 and abs(qy - ty) < min_gap:
        ty = qy + min_gap
    placed.append((px, ty))
    moved = abs(ty - py) > 0.01
    ax.annotate(labels[i], xy=(px, py), xycoords="axes fraction", xytext=(px + dx, ty),
                textcoords="axes fraction", fontsize=fontsize, va="center",
                arrowprops=dict(arrowstyle="-", color=GREY, lw=0.4) if moved else None)

def end_labels(ax, items, x, min_gap=0.045, fontsize=7.5):
  """Spreads out labels (overlaps) at the right end of lines.

  items : [(text, y_value, color), ...]
  """
  lo, hi = ax.get_ylim()
  log = ax.get_yscale() == "log"
  f = np.log10 if log else (lambda v: v)
  finv = (lambda v: 10 ** v) if log else (lambda v: v)
  gap = min_gap * (f(hi) - f(lo))
  ordered = sorted((f(yv), text, color) for text, yv, color in items)
  ys = []
  for yv, _, _ in ordered:
    ys.append(max(yv, ys[-1] + gap) if ys else yv)
  for (_, text, color), yp in zip(ordered, ys):
    ax.text(x, finv(yp), text, color=color, va="center", fontsize=fontsize)


set_thesis_style()

# Table Writing Helpers
def write_table(df, name, directory=None, caption=None, label=None):
  """Writes `df` to <directory>/<name>.csv

  directory : defaluts to THESIS_TABLES_DIR
  """
  directory = Path(directory or THESIS_TABLES_DIR)
  print(f"Writing to: {directory}/{name}.csv")
  df.to_csv(directory / f"{name}.csv", index=False)

def fmt(value, decimals=2, pct=False):
  """Number -> string with fixed decimals (pct=True multiplies by 100)."""
  if value is None or (isinstance(value, float) and np.isnan(value)):
    return "--"
  return f"{value * 100 if pct else value:,.{decimals}f}"

# Create Analysis Panel

**These steps are only necessary when running the code for the first time**, without an analysis panel and with only the raw, source CSV files. Otherwise, if this is a re-run, or the HS4 panel was downloaded seaparately, code can be run directly from *Thesis Results* (provided the folder structure in *LEGEND* is followed.

## Merge source parquet/CSV files

Download the HGL source dataset from: [http://doi.org/10.7910/DVN/5NGVOB](http://doi.org/10.7910/DVN/5NGVOB)

The dataset will be split in different CSV files, which this function merges into a single parquet file.

In [ ]:
def merge_source_dataset(con, source_files_dir=SOURCE_DB_DIR):
  """
  source_files_dir  : The directory where raw .csv HGL dataset files reside
  """
  raw_filenames = [file.stem for file in source_files_dir.glob('*.csv')]

  q = f"COPY (\n"
  for i in range(len(raw_filenames)):
    filename = raw_filenames[i]
    filepath = source_files_dir / f"{filename}.csv"
    q += f"""
      SELECT * FROM '{filepath}'
    """
    q += f"\nUNION ALL " if i < len(raw_filenames)-1 else ""
  q += f") TO '{MERGED_DB_PATH}' (FORMAT 'PARQUET');"

  con.execute(q)
  print(f"Successful merged copy at [{MERGED_DB_PATH}]")
  print("\nSanity Check: Row Count\n")
  total_merged_rows = con.execute(f"SELECT COUNT(*) FROM '{MERGED_DB_PATH}'").fetchone()[0]
  total_rows = 0
  for i in range(len(raw_filenames)):
    filename = raw_filenames[i]
    filepath = source_files_dir / f"{filename}.csv"
    rows = con.execute(f"SELECT COUNT(*) FROM '{filepath}'").fetchone()[0]
    total_rows += rows
    print(f" [{i}] {filename}: {rows} rows\n")

  print(f"\nTotal Rows (raw files): {total_rows}")
  print(f"\nTotal Rows (merged file): {total_merged_rows}")

  assert total_rows == total_merged_rows, "Merging did not produce the correct amount of rows"
  return MERGED_DB_PATH

## Create enriched table
Creates a physical DuckDB table for the enriched data, used in later steps, assuming the existence of a merged dataset as created by the preceding function. Adds various quality and diagnostic flags to trade data (not all of which are selected for the final HS4 Panel):

- `is_numeric_hs6`: Rows where 'commoditycode' is a 6 digit HS code
- `hs4`: Extracts the first 4 digits of found numeric HS6 codes
- `is_forest`: Flags forest products based on their HS4 code
- `has_exporter_report`: Indicates if exporter's reported value is present
- `has_importer_report`: Indicates if importer's reported value is present
- `reporting_case`: Categorizes trade records into 'both', 'exporter_only', 'importer_only', or 'neither' based on presence of exporter and importer reports
- `raw_discrepancy_harvard`: Calculates raw discrepancy measure as per the Harvard methodology ([0,1]) between exporter and importer values when both are available and their sum is positive
- `raw_gap_abs`: Absolute difference between exporter and importer values when both are present
- `final_vs_exporter_pct`: Percentage difference between the final reconciled value ('value_final') and the exporter's reported value ('value_exporter'), if the latter is not zero
- `final_vs_importer_pct`: Percentage difference between the final reconciled value ('value_final') and the importer's reported value ('value_importer'), if the latter is not zero

In [ ]:
forest_list_sql = ", ".join([f"'{x}'" for x in list(HS4_NAMES.keys())])

def enrich_raw_database(con, db_dir=DB_DIR, merged_db_filename=MERGED_DB_NAME):
  """
  db_dir              : The directory with the complete merged dataset
  merged_db_filename  : The name of the merged .parquet dataset, without extension
  """
  db_path = f"{db_dir}/{merged_db_filename}.parquet"
  q = f"""
  CREATE OR REPLACE TABLE enriched_harvard AS
  SELECT
    year,
    exporter,
    importer,
    commoditycode,
    value_exporter,
    value_importer,
    value_final,

    regexp_full_match(commoditycode, '^[0-9]{{6}}$') AS is_numeric_hs6,

    CASE
      WHEN regexp_full_match(commoditycode, '^[0-9]{{6}}$')
      THEN SUBSTR(commoditycode, 1, 4)
      ELSE NULL
    END AS hs4,

    CASE
      WHEN regexp_full_match(commoditycode, '^[0-9]{{6}}$')
          AND SUBSTR(commoditycode, 1, 4) IN ({forest_list_sql})
      THEN TRUE
      ELSE FALSE
    END AS is_forest,

    value_exporter IS NOT NULL AND value_exporter <> 0 AS has_exporter_report,
    value_importer IS NOT NULL AND value_importer <> 0 AS has_importer_report,

    CASE
      WHEN has_exporter_report AND has_importer_report THEN 'both'
      WHEN has_exporter_report AND NOT has_importer_report THEN 'exporter_only'
      WHEN NOT has_exporter_report AND has_importer_report THEN 'importer_only'
      ELSE 'neither'
    END AS reporting_case,

    CASE
      WHEN value_exporter IS NOT NULL
        AND value_importer IS NOT NULL
        AND (value_exporter + value_importer) > 0
      THEN ABS(value_exporter - value_importer) / (value_exporter + value_importer)
      ELSE NULL
    END AS raw_discrepancy_harvard,

    CASE
      WHEN value_exporter IS NOT NULL AND value_importer IS NOT NULL
      THEN ABS(value_exporter - value_importer)
      ELSE NULL
    END AS raw_gap_abs,

    CASE
      WHEN value_exporter IS NOT NULL AND value_exporter <> 0
      THEN ((value_final - value_exporter) / value_exporter) * 100
      ELSE NULL
    END AS final_vs_exporter_pct,

    CASE
      WHEN value_importer IS NOT NULL AND value_importer <> 0
      THEN ((value_final - value_importer) / value_importer) * 100
      ELSE NULL
    END AS final_vs_importer_pct

  FROM read_parquet('{db_path}');
  """

  con.execute(q)
  print(f"Created view `enriched_harvard` from merged database at [{db_path}]")


## Create Forest Edge Table
Filter the dataset to include only rows identified as forest products (i.e., marked with `is_forest` flag). Assumes the existence of view `enriched_harvard` created by the preceding function, and creates new view `forest_edges`.

If future revisions want to amend the list of products, or adapt the analysis to different sectors, simply change the HS4 values (and names) in `HS4_NAMES`.

In [ ]:
def forest_table(con):
  q = f"""
    CREATE OR REPLACE TABLE forest_edges AS
    SELECT
      year,
      exporter,
      importer,
      commoditycode,
      hs4,
      value_exporter,
      value_importer,
      value_final,
      has_exporter_report,
      has_importer_report,
      reporting_case,
      raw_discrepancy_harvard,
      raw_gap_abs,
      final_vs_exporter_pct,
      final_vs_importer_pct
    FROM enriched_harvard
    WHERE is_forest = TRUE;
    """
  con.execute(q)
  print("Created view `forest_edges` from enriched view `enriched_harvard`")

## Validate duplicates
This function runs before building the final aggregated HS4 analysis panel to check whether there is a single record for each `YEAR x EXPORTER x IMPORTER x HS6` combination.

In [ ]:
def check_duplicates(con):
  con.execute("""
    CREATE OR REPLACE TABLE duplicate_rows AS
    SELECT
      year, exporter, importer, commoditycode, COUNT(*) AS n_rows
    FROM forest_edges
    GROUP BY year, exporter, importer, commoditycode
    HAVING COUNT(*) > 1;
  """)

  dup_count = con.execute("SELECT COUNT(*) FROM duplicate_rows").fetchone()[0]
  print(f"Duplicate key count: {dup_count}")
  return dup_count

## Build HS4 Analysis Panel

This section builds the `YEAR x EXPORTER x IMPORTER x HS4` analysis panel from the `forest_edges` view generated in the previous function. **All results will follow from this panel.**

In [ ]:
def build_panel(con, panel_path=PANEL_PATH):
  """
  path  : the location where the final HS4 panel will be saved, used by `connect_panel` for all outputs
  """
  q = f"""
  COPY (
    WITH hs6 AS (
      SELECT year, exporter, importer, hs4, commoditycode,
        value_exporter, value_importer, value_final,

        CASE
          WHEN value_exporter IS NOT NULL AND value_exporter > 0
          THEN 1 ELSE 0
        END AS has_exporter,

        CASE
          WHEN value_importer IS NOT NULL AND value_importer > 0
          THEN 1 ELSE 0
        END AS has_importer

      FROM forest_edges
      WHERE hs4 IS NOT NULL AND exporter IS NOT NULL AND importer IS NOT NULL AND exporter <> importer
    ),

    aggregated AS (
      SELECT year, exporter, importer, hs4,
        -- E_xmcy
        SUM(
          CASE
            WHEN has_exporter = 1 THEN value_exporter
            ELSE 0
          END
        ) AS E,

        -- I_xmcy
        SUM(
          CASE
            WHEN has_importer = 1 THEN value_importer
            ELSE 0
          END
        ) AS I,

        -- F_xmcy
        SUM(COALESCE(value_final, 0)) AS F,

        COUNT(*) AS n_hs6_rows,
        COUNT(DISTINCT commoditycode) AS n_hs6_products,

        SUM(
          CASE
            WHEN has_exporter = 1 AND has_importer = 1
            THEN 1 ELSE 0
          END
        ) AS n_both_hs6,

        SUM(
          CASE
            WHEN has_exporter = 1 AND has_importer = 0
            THEN 1 ELSE 0
          END
        ) AS n_exporter_only_hs6,

        SUM(
          CASE
            WHEN has_exporter = 0 AND has_importer = 1
            THEN 1 ELSE 0
          END
        ) AS n_importer_only_hs6

      FROM hs6
      GROUP BY year, exporter, importer, hs4
    )

    SELECT year, exporter, importer, hs4,
      E, I, F,

      n_hs6_rows, n_hs6_products,

      n_both_hs6, n_exporter_only_hs6, n_importer_only_hs6,

      n_both_hs6 * 1.0 / n_hs6_rows AS share_both_hs6,
      n_exporter_only_hs6 * 1.0 / n_hs6_rows AS share_exporter_only_hs6,
      n_importer_only_hs6 * 1.0 / n_hs6_rows AS share_importer_only_hs6,

      E > 0 AS has_exporter_report, I > 0 AS has_importer_report,

      CASE
        WHEN E > 0 AND I > 0 THEN 'both'
        WHEN E > 0 AND I <= 0 THEN 'exporter_only'
        WHEN E <= 0 AND I > 0 THEN 'importer_only'
        ELSE 'neither'
      END AS reporting_case,

      -- Equation 3.4: absolute normalized discrepancy D
      CASE
        WHEN E > 0 AND I > 0
        THEN ABS(E - I) / (E + I)
        ELSE NULL
      END AS D,

      -- Equation 3.5: signed normalized discrepancy S
      CASE
        WHEN E > 0 AND I > 0
        THEN (E - I) / (E + I)
        ELSE NULL
      END AS S,

      -- Equation 3.6: absolute discrepancy G
      CASE
        WHEN E > 0 AND I > 0
        THEN ABS(E - I)
        ELSE NULL
      END AS G

    FROM aggregated
  ) TO '{panel_path}' (FORMAT PARQUET, COMPRESSION ZSTD);
  """
  con.execute(q)
  print(f"Created [{panel_path}]")

In [ ]:
# Panel Printout
def hs4_panel_summary(con, panel_path=PANEL_PATH):
  print(con.execute(f"""
    SELECT
      COUNT(*) AS n_rows,
      COUNT(DISTINCT year) AS n_years,
      COUNT(DISTINCT exporter) AS n_exporters,
      COUNT(DISTINCT importer) AS n_importers,
      COUNT(DISTINCT hs4) AS n_hs4,
      COUNT(D) AS n_valid_discrepancies
    FROM read_parquet('{panel_path}')
  """).fetchdf().to_string())

# Thesis Results

Every figure and table in Chapter 4 and the appendices of the thesis is produced by one function in this section. All functions read the `YEAR x EXPORTER x IMPORTER x HS4` analysis panel `panel` through DuckDB, return the figure (or table) together with the data behind it, and save the output to `THESIS_FIGURES_DIR` or `THESIS_TABLES_DIR` (if `SAVE_OUTPUTS` is set to `True`).

Customise a figure by changing the function's parameters (years, sample threshold, axis limits, number of labelled countries, ...).

## DuckDB Retrieval Setup

Sets up data retrieval, using `connect_panel()` to open a DuckDB connection in memory with a view `panel` on the HS4 analysis panel **already created and in path `panel_path`**.<br>

`sample_filter()` builds the SQL `WHERE` clause that defines a sample with:
* `min_flow=0` gives all two-sided flows (the full sample of Sections 4.1-4.3),
* `min_flow=MIN_FLOW` gives the analysis sample used from Section 4.4 onwards.

In [ ]:
def connect_panel(panel_path=PANEL_PATH, memory_limit="2GB", threads=4, real_usd=REAL_USD, base_year=PRICE_BASE_YEAR):
  """DuckDB connection with a view `panel` (the HS4 panel plus an `hs2` chapter column).

  real_usd : calculate E, I, F and G in constant `base_year` USD (CPI-U), where `F_nominal` keeps the
             reported current-USD reconciled value (for the sample threshold and size classes).
  """
  con = duckdb.connect()
  con.execute(f"SET memory_limit = '{memory_limit}'")
  con.execute(f"SET threads = {threads}")
  con.execute("SET preserve_insertion_order = false")
  # Default to multiplier for base year, otherwise multiplier is set to 1 (i.e., no adjustment)
  deflator = deflator_table(base_year) if real_usd else deflator_table(base_year).assign(to_real=1.0)
  con.register("deflator", deflator)
  # Select all columns, but change E, I, F, and G to constant USD (if real_usd is True)
  #   and create _nominal counterparts
  con.execute(f"""
    CREATE OR REPLACE VIEW panel AS
    SELECT p.* REPLACE (p.E * d.to_real AS E, p.I * d.to_real AS I, p.F * d.to_real AS F, p.G * d.to_real AS G),
           p.F AS F_nominal, p.E AS E_nominal, p.I AS I_nominal, left(p.hs4, 2) AS hs2
    FROM read_parquet('{Path(panel_path).as_posix()}') p JOIN deflator d USING (year)
  """)
  return con

def value_unit(con):
  """For axis labels, use "constant 2024 USD" if connection uses deflated values, otherwise "current USD"."""
  # If F = F_nominal for the smallest year in the panel, then no adjustment is made
  real = con.execute("SELECT bool_or(F <> F_nominal) FROM panel WHERE year = (SELECT min(year) FROM panel) AND F > 0").fetchone()[0]
  return f"constant {PRICE_BASE_YEAR} USD" if real else "current USD"

def sample_filter(min_flow=0, years=STUDY_YEARS, exclude_hs4=(), two_sided=True, extra=None):
  """Forms the SQL query to include only the sample selection of the panel (plugged in a WHERE clause).

  two_sided   : keep only flows reported by both partners (D, S, and G are only calculated there)
  min_flow    : minimum reconciled value F_nominal in USD (0 = no threshold)
  exclude_hs4 : headings to leave out, e.g. FLAGGED_HS4
  extra       : any additional SQL condition, e.g. FLAGGED_FLOWS
  """
  conditions = [f"year BETWEEN {years[0]} AND {years[1]}"]
  if two_sided:
    conditions.append("reporting_case = 'both'")
  if min_flow:
    conditions.append(f"F_nominal >= {min_flow}")
  if exclude_hs4:
    conditions.append("hs4 NOT IN (" + ", ".join(f"'{h}'" for h in exclude_hs4) + ")")
  if extra:
    conditions.append(f"({extra})")
  return " AND ".join(conditions)

def query(con, sql):
  """Run an SQL query through DuckDB and return a pandas DataFrame."""
  return con.execute(sql).df()

## Table 2.1: Non-reporting countries per year

A country *reports* in a year if it declares at least one forest-product flow (i.e. exports `E > 0` or imports `I > 0`). A *non-reporting* country appears in the panel only through partner reports.

In [ ]:
def tab_2_1_non_reporting(con, years=STUDY_YEARS, name="tab_2_1_non_reporting", save=SAVE_OUTPUTS):
  data = query(con, f"""
    WITH roles AS (
      SELECT year, exporter AS country, E > 0 AS reports_exports, FALSE AS reports_imports
      FROM panel WHERE year BETWEEN {years[0]} AND {years[1]}
      UNION ALL
      SELECT year, importer, FALSE, I > 0
      FROM panel WHERE year BETWEEN {years[0]} AND {years[1]}
    ),
    country_year AS (
      SELECT year, country, bool_or(reports_exports) AS rx, bool_or(reports_imports) AS rm
      FROM roles GROUP BY year, country
    )
    SELECT year,
      count(*)                              AS countries_in_network,
      sum((rx OR rm)::INT)                  AS reporting,
      sum((NOT rx)::INT)                    AS no_export_reports,
      sum((NOT rm)::INT)                    AS no_import_reports,
      sum((NOT rx AND NOT rm)::INT)         AS non_reporting
    FROM country_year GROUP BY year ORDER BY year
  """)
  data["non_reporting_pct"] = 100 * data["non_reporting"] / data["countries_in_network"]
  table = data.assign(non_reporting_pct=data["non_reporting_pct"].map(lambda v: f"{v:.1f}"))
  if save:
    write_table(table, name)
  return table, data

## Figure 4.1: Reporting completeness

*(a)* share of `year x exporter x importer x HS4` flows by reporting case *(b)* share of the annual reconciled value F of one-sided flows. In panel (b) the country that is not reporting on the largest one-sided value is labelled above each year through parameter `annotate_years`.

In [ ]:
def fig_4_1_reporting_cases(con, years=STUDY_YEARS, annotate_years=(2011, 2022, 2024), figsize=(10, 4),
                            value_ylim=(0, 15), name="fig_4_1_reporting_cases", save=SAVE_OUTPUTS):
  cases = ["both", "exporter_only", "importer_only", "neither"]
  data = query(con, f"""
    SELECT year, reporting_case, count(*) AS n, sum(F) AS F
    FROM panel WHERE {sample_filter(years=years, two_sided=False)}
    GROUP BY year, reporting_case
  """)
  n = data.pivot(index="year", columns="reporting_case", values="n").reindex(columns=cases).fillna(0)
  value = data.pivot(index="year", columns="reporting_case", values="F").reindex(columns=cases).fillna(0)

  n_share = 100 * n.div(n.sum(axis=1), axis=0)
  value_share = 100 * value.div(value.sum(axis=1), axis=0)

  silent_countries = query(con, f"""
    SELECT * FROM (
      SELECT year, CASE WHEN reporting_case = 'exporter_only' THEN importer ELSE exporter END AS silent_country,
             sum(F) AS F
      FROM panel
      WHERE reporting_case IN ('exporter_only', 'importer_only') AND year IN ({", ".join(map(str, annotate_years))})
      GROUP BY year, silent_country
    )
    QUALIFY row_number() OVER (PARTITION BY year ORDER BY F DESC) = 1
  """) if annotate_years else pd.DataFrame(columns=["year", "silent_country", "F"])

  fig, axes = plt.subplots(1, 2, figsize=figsize)
  ax = axes[0]
  ax.stackplot(n_share.index, n_share.T.values, colors=[CASE_COLORS[c] for c in cases],
               labels=[CASE_LABELS[c] for c in cases], lw=0)
  ax.set(xlim=years, ylim=(0, 100), xlabel="Year", ylabel="Share of flows (%)")
  ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
  panel_label(ax, "a")

  ax = axes[1]
  one_sided = value_share["exporter_only"] + value_share["importer_only"]
  for case in ["exporter_only", "importer_only"]:
    ax.plot(value_share.index, value_share[case], color=CASE_COLORS[case], lw=1.8, label=CASE_LABELS[case])
  ax.plot(one_sided.index, one_sided, color="black", lw=1.0, ls="--", label="Both one-sided cases")
  for _, row in silent_countries.iterrows():
    ax.annotate(row["silent_country"], xy=(row["year"], one_sided[row["year"]]), xytext=(0, 6),
                textcoords="offset points", ha="center", fontsize=7)
  ax.set(xlim=years, ylim=value_ylim, xlabel="Year", ylabel="Share of annual reconciled value F (%)")
  ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
  panel_label(ax, "b")
  fig.tight_layout()
  if save:
    save_figure(fig, name)

  summary = pd.concat({"n": n, "F": value, "n_share": n_share, "F_share": value_share}, axis=1)
  return fig, summary, silent_countries

## Figure 4.2: Annual distribution of D

Median and inter-quartile range (IQR) of `D` per year, the 90th percentile, the unweighted mean, and value-weighted mean D<sub>w</sub>:
```
D_w = ∑|E - I| / ∑(E + I)
```

In [ ]:
def annual_discrepancy_stats(con, min_flow=0, years=STUDY_YEARS, exclude_hs4=(), extra=None):
  """Per-year distribution of D, S and G for a sample of two-sided flows."""
  return query(con, f"""
    SELECT year, count(*) AS n_flows,
      quantile_cont(D, 0.10) AS D_p10, quantile_cont(D, 0.25) AS D_p25, quantile_cont(D, 0.50) AS D_p50,
      quantile_cont(D, 0.75) AS D_p75, quantile_cont(D, 0.90) AS D_p90, avg(D) AS D_mean,
      sum(G) / sum(E + I) AS D_w,
      quantile_cont(S, 0.10) AS S_p10, quantile_cont(S, 0.25) AS S_p25, quantile_cont(S, 0.50) AS S_p50,
      quantile_cont(S, 0.75) AS S_p75, quantile_cont(S, 0.90) AS S_p90, avg(S) AS S_mean,
      sum(E - I) / sum(E + I) AS S_w, avg((S < 0)::INT) AS share_S_negative,
      sum(G) AS G_sum, sum(F) AS F_sum, quantile_cont(G, 0.5) AS G_p50
    FROM panel WHERE {sample_filter(min_flow, years, exclude_hs4, extra=extra)}
    GROUP BY year ORDER BY year
  """).set_index("year")


def fig_4_2_annual_D(con, min_flow=0, years=STUDY_YEARS, ylim=(0, 1), show_p90=True, show_mean=True,
                     figsize=(7.5, 3.8), name="fig_4_2_annual_D", save=SAVE_OUTPUTS):
  data = annual_discrepancy_stats(con, min_flow, years)
  fig, ax = plt.subplots(figsize=figsize)
  ax.fill_between(data.index, data["D_p25"], data["D_p75"], color=FOCUS, alpha=0.18, lw=0)
  ax.plot(data.index, data["D_p50"], color=FOCUS, lw=2.2, marker="o", ms=3)
  labels = [("Median", data["D_p50"].iloc[-1], FOCUS), ("IQR (P25-P75)", data["D_p75"].iloc[-1], FOCUS),
            ("Value-weighted mean $D_w$", data["D_w"].iloc[-1], ACCENT)]
  ax.plot(data.index, data["D_w"], color=ACCENT, lw=2)
  if show_p90:
    ax.plot(data.index, data["D_p90"], color=FOCUS, lw=1, ls="--")
    labels.append(("P90", data["D_p90"].iloc[-1], FOCUS))
  if show_mean:
    ax.plot(data.index, data["D_mean"], color=GREY, lw=1.2, ls=":")
    labels.append(("Mean", data["D_mean"].iloc[-1], GREY))
  ax.set(xlim=years, ylim=ylim, xlabel="Year", ylabel="Absolute normalized discrepancy D")
  end_labels(ax, labels, years[1] + 0.4)
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, data

## Figure 4.3: Direction and magnitude of discrepancies over time

*(a)* distribution of the signed discrepancy S *(b)* total reconciled value and total absolute discrepancy `G` *(c)* total `G` as a share of total `F`.

In [ ]:
def fig_4_3_annual_S_G(con, min_flow=0, years=STUDY_YEARS, s_ylim=(-1, 1), ratio_ylim=(0, 50),
                       figsize=(12, 3.8), name="fig_4_3_annual_S_G", save=SAVE_OUTPUTS):
  data = annual_discrepancy_stats(con, min_flow, years)
  fig, axes = plt.subplots(1, 3, figsize=figsize)
  # Panel (a)
  ax = axes[0]
  ax.axhline(0, color="black", lw=0.6)
  ax.fill_between(data.index, data["S_p10"], data["S_p90"], color=FOCUS, alpha=0.08, lw=0, label="P10-P90")
  ax.fill_between(data.index, data["S_p25"], data["S_p75"], color=FOCUS, alpha=0.22, lw=0, label="IQR")
  ax.plot(data.index, data["S_p50"], color=FOCUS, lw=2, marker="o", ms=3, label="Median")
  ax.set(xlim=years, ylim=s_ylim, xlabel="Year", ylabel="Signed normalized discrepancy S")

  ax.text(0.02, 0.97, "exporter reports more", transform=ax.transAxes, va="top", fontsize=7, color=GREY)
  ax.text(0.02, 0.03, "importer reports more", transform=ax.transAxes, va="bottom", fontsize=7, color=GREY)
  ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
  panel_label(ax, "a")

  # Panel (b)
  ax = axes[1]
  ax.plot(data.index, data["F_sum"] / 1e9, color=FOCUS, lw=2)
  ax.plot(data.index, data["G_sum"] / 1e9, color=CONTRAST, lw=2)
  ax.set(xlim=years, ylim=(0, None), xlabel="Year", ylabel=f"Billion {value_unit(con)}")
  end_labels(ax, [("Reconciled value F", data["F_sum"].iloc[-1] / 1e9, FOCUS),
                  ("Absolute discrepancy G", data["G_sum"].iloc[-1] / 1e9, CONTRAST)], years[1] + 0.4)
  panel_label(ax, "b")

  # Panel (c)
  ax = axes[2]
  ratio = 100 * data["G_sum"] / data["F_sum"]
  ax.plot(data.index, ratio, color=CONTRAST, lw=2, marker="o", ms=3)
  ax.set(xlim=years, ylim=ratio_ylim, xlabel="Year", ylabel="Total G / total F (%)")
  panel_label(ax, "c")
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, data.assign(G_over_F_pct=ratio)

## Figure 4.4: Discrepancy and flow size

*(a)* annual median D per year within bands of the reconciled value `F` *(b)* Median and IQR of D against F when aggregating all years, dashed line marks the threshold used for the analysis sample `MIN_FLOW`.

In [ ]:
def size_class_sql(breaks):
  """SQL CASE clause to assign each flow to a different category of nominal F (e.g. labels as '1k-10k')."""
  edges = list(breaks)
  # If F_nominal = 0 or lower than minimum break:
  sql = ["CASE WHEN F_nominal = 0 THEN 'F = 0'", f"WHEN F_nominal < {edges[0]} THEN '<{usd(edges[0])}'"]
  # Zip together list without last element and list without first element to get the band
  for lo, hi in zip(edges[:-1], edges[1:]):
    sql.append(f"WHEN F_nominal < {hi} THEN '{usd(lo)}-{usd(hi)}'")
  sql.append(f"ELSE '>={usd(edges[-1])}' END")
  labels = ["F = 0", f"<{usd(edges[0])}"] + [f"{usd(a)}-{usd(b)}" for a, b in zip(edges[:-1], edges[1:])] + [f">={usd(edges[-1])}"]
  return " ".join(sql), labels


def fig_4_4_D_by_flow_size(con, years=STUDY_YEARS, size_breaks=(1e3, 1e4, 1e5, 1e6, 1e7, 1e8), bins_per_decade=5,
                        mark_threshold=MIN_FLOW, break_year=2000, figsize=(11, 4), name="fig_4_4_D_by_flow_size",
                        save=SAVE_OUTPUTS):
  case_sql, classes = size_class_sql(size_breaks)
  by_class = query(con, f"""
    SELECT year, {case_sql} AS size_class, count(*) AS n_flows, median(D) AS D_p50
    FROM panel WHERE {sample_filter(0, years)} GROUP BY ALL
  """)
  by_bin = query(con, f"""
    SELECT floor(log10(F_nominal) * {bins_per_decade}) / {bins_per_decade} AS log10_F, count(*) AS n_flows,
      quantile_cont(D, 0.25) AS D_p25, median(D) AS D_p50, quantile_cont(D, 0.75) AS D_p75
    FROM panel WHERE {sample_filter(0, years)} AND F_nominal > 0
    GROUP BY ALL HAVING count(*) >= 50 ORDER BY log10_F
  """)
  colors = dict(zip(classes, plt.get_cmap("viridis_r")(np.linspace(0.05, 0.95, len(classes)))))

  fig, axes = plt.subplots(1, 2, figsize=figsize)

  # Panel (a)
  ax = axes[0]
  labels = []
  for size_class in classes:
    d = by_class[by_class["size_class"] == size_class].sort_values("year")
    ax.plot(d["year"], d["D_p50"], color=colors[size_class], lw=1.8)
    labels.append((size_class, d["D_p50"].iloc[-1], colors[size_class]))
  if break_year:
    # Mark half way befor break year
    ax.axvline(break_year - 0.5, color=GREY, lw=0.8, ls="--")
  ax.set(xlim=years, ylim=(0, 1), xlabel="Year", ylabel="Median D")
  end_labels(ax, labels, years[1] + 0.4)
  panel_label(ax, "a")

  # Panel (b)
  ax = axes[1]
  x = 10 ** (by_bin["log10_F"] + 0.5 / bins_per_decade)
  ax.fill_between(x, by_bin["D_p25"], by_bin["D_p75"], color=FOCUS, alpha=0.2, lw=0, label="IQR")
  ax.plot(x, by_bin["D_p50"], color=FOCUS, lw=2, label="Median")
  if mark_threshold:
    ax.axvline(mark_threshold, color=CONTRAST, lw=1, ls="--")
    ax.text(mark_threshold * 1.15, 0.95, f"F = {usd(mark_threshold)} USD", color=CONTRAST, fontsize=7, va="top")
  ax.set_xscale("log")
  ax.xaxis.set_major_formatter(USD_FORMATTER)
  ax.set(ylim=(0, 1), xlabel="Reconciled flow value F (nominal USD, log scale)", ylabel="D")
  ax.legend(loc="upper right", bbox_to_anchor=(1, 0.88))
  panel_label(ax, "b")
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, by_class, by_bin

## Table 4.1: Discrepancy statistics with specific minimum trade flow sizes

In [ ]:
def tab_4_1_min_flow_sensitivity(con, thresholds=(0, 1e3, 1e4, 1e5, 1e6), years=STUDY_YEARS, break_year=2000,
                                   name="tab_4_1_min_flow_sensitivity", save=SAVE_OUTPUTS):
  total = query(con, f"SELECT count(*) AS n, sum(F) AS F FROM panel WHERE {sample_filter(0, years)}").iloc[0]
  rows = []
  for threshold in thresholds:
    r = query(con, f"""
      SELECT count(*) AS n, sum(F) AS F,
        median(D) FILTER (WHERE year < {break_year})  AS D_p50_before,
        median(D) FILTER (WHERE year >= {break_year}) AS D_p50_after,
        median(D) AS D_p50, sum(G) / sum(E + I) AS D_w, avg((D >= 0.99)::INT) AS share_D99
      FROM panel WHERE {sample_filter(threshold, years)}
    """).iloc[0]
    rows.append({"threshold": threshold, "flows_kept_pct": 100 * r["n"] / total["n"],
                 "value_kept_pct": 100 * r["F"] / total["F"], **r.to_dict()})
  data = pd.DataFrame(rows)
  table = pd.DataFrame({
    "Minimum F (USD)": ["none" if t == 0 else f"{t:,.0f}" for t in data["threshold"]],
    "Flows kept (%)": data["flows_kept_pct"].map(lambda v: fmt(v, 1)),
    "Value kept (%)": data["value_kept_pct"].map(lambda v: fmt(v, 2)),
    f"Median D {years[0]}-{break_year - 1}": data["D_p50_before"].map(lambda v: fmt(v, 3)),
    f"Median D {break_year}-{years[1]}": data["D_p50_after"].map(lambda v: fmt(v, 3)),
    "Value-weighted $D_w$": data["D_w"].map(lambda v: fmt(v, 3)),
    r"$D>=0.99$ (%)": data["share_D99"].map(lambda v: fmt(v, 1, pct=True)),
  })
  if save:
    write_table(table, name)
  return table, data

## Figure 4.5: Where does the variation in discrepancy come from?

One-way analysis of variance of D for each grouping factor. The bars show the adjusted share of variance explained: ϵ<sup>2</sup> = (SS<sub>between</sub> - (k-1)*MS<sub>within</sub>)&nbsp;&frasl;&nbsp;SS<sub>total</sub>, which adjuists
for the number of groups `k` to ensure a grouping with many small groups are not inflated.

In [ ]:
VARIANCE_FACTORS = {
  "Year": "year",
  "HS chapter": "hs2",
  "HS4 heading": "hs4",
  "Heading x year": "hs4, year",
  "Exporter": "exporter",
  "Importer": "importer",
  "Country pair": "exporter, importer",
  "Exporter x chapter": "exporter, hs2",
  "Importer x chapter": "importer, hs2",
  "Exporter x heading": "exporter, hs4",
  "Importer x heading": "importer, hs4",
  "Country pair x chapter": "exporter, importer, hs2",
  "Country pair x heading": "exporter, importer, hs4",
}


def variance_explained(con, var="D", factors=VARIANCE_FACTORS, min_flow=MIN_FLOW, years=STUDY_YEARS,
                       exclude_hs4=(), extra=None):
  """eta2 and epsilon2 of param `var` for every grouping factor (one-way ANOVA)."""
  where = sample_filter(min_flow, years, exclude_hs4, extra=extra)
  n, mean, ss_total = con.execute(f"SELECT count(*), avg({var}), var_pop({var}) * count(*) FROM panel WHERE {where}").fetchone()
  rows = []
  # For each factor:
  for label, columns in factors.items():
    # Calculate # of groups (k) and between-group SS
    k, ss_between = con.execute(f"""
      SELECT count(*), sum(n_g * pow(mean_g - ({mean}), 2))
      FROM (SELECT count(*) AS n_g, avg({var}) AS mean_g FROM panel WHERE {where} GROUP BY {columns}) AS groups
    """).fetchone()
    # Calculate NSE from within-group SS
    ms_within = (ss_total - ss_between) / (n - k)
    rows.append({"factor": label, "groups": k, "eta2": ss_between / ss_total,
                 "epsilon2": (ss_between - (k - 1) * ms_within) / ss_total, "n_flows": n})
  return pd.DataFrame(rows)


def fig_4_5_variance_decomposition(con, var="D", samples=None, factors=VARIANCE_FACTORS, xlim=(0, 40),
                                   figsize=(7.5, 4.2), name="fig_4_5_variance_decomposition", save=SAVE_OUTPUTS):
  samples = samples or {f"Analysis sample (F >= {usd(MIN_FLOW)} USD)": MIN_FLOW, "All two-sided flows": 0}
  data = pd.concat([variance_explained(con, var, factors, min_flow).assign(sample=label)
                    for label, min_flow in samples.items()])
  order = data[data["sample"] == list(samples)[0]].sort_values("epsilon2")["factor"].tolist()
  colors = [FOCUS, ACCENT, GREY, CONTRAST]
  fig, ax = plt.subplots(figsize=figsize)
  y = np.arange(len(order))
  for j, label in enumerate(samples):
    # Filter main df
    d = data[data["sample"] == label].set_index("factor").loc[order]
    # Alternate offsets to avoif overlap
    offset = (j - (len(samples) - 1) / 2) * 0.28
    # h-line from 0 to 100*epsilon2 for all groupings
    ax.hlines(y + offset, 0, 100 * d["epsilon2"], color=colors[j], lw=1, alpha=0.5)
    # add marker plot 'o' at the end of each h-line
    ax.plot(100 * d["epsilon2"], y + offset, "o", color=colors[j], ms=5, label=label)
  groups = data[data["sample"] == list(samples)[0]].set_index("factor")["groups"]
  ax.set_yticks(y, [f"{f}  (k = {groups[f]:,})" for f in order])
  ax.set(xlim=xlim, xlabel=f"Variance of {var} explained (adjusted $\\epsilon^2$, %)")
  ax.legend(loc="lower right")
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, data

## HS4-level Statistics

Used in Figures 4.6-4.7, Table 4.2, and Table C.1

In [ ]:
def heading_stats(con, min_flow=MIN_FLOW, years=STUDY_YEARS, exclude_hs4=(), extra=None):
  """One row per HS4 heading: D & S statistics, plus total F, G, and G/F."""
  sample = query(con, f"""
    SELECT hs2, hs4, count(*) AS n_flows, sum(F) AS F_sample,
      quantile_cont(D, 0.10) AS D_p10, quantile_cont(D, 0.25) AS D_p25, median(D) AS D_p50,
      quantile_cont(D, 0.75) AS D_p75, quantile_cont(D, 0.90) AS D_p90, sum(G) / sum(E + I) AS D_w,
      quantile_cont(S, 0.25) AS S_p25, median(S) AS S_p50, quantile_cont(S, 0.75) AS S_p75,
      sum(E - I) / sum(E + I) AS S_w, sum(G) AS G_sum, avg((D > 0.5)::INT) AS share_D_gt05
    FROM panel WHERE {sample_filter(min_flow, years, exclude_hs4, extra=extra)}
    GROUP BY hs2, hs4
  """)
  # Total F and bilateral reporting share for all flows, except where reporting_case = 'neither'
  totals = query(con, f"""
    SELECT hs4, sum(F) AS F_total, avg((reporting_case = 'both')::INT) AS two_sided_share
    FROM panel WHERE year BETWEEN {years[0]} AND {years[1]} AND reporting_case <> 'neither'
    GROUP BY hs4
  """)
  # Merge together using hs4 as key
  data = sample.merge(totals, on="hs4")
  data["G_over_F"] = data["G_sum"] / data["F_sample"]
  data["name"] = data["hs4"].map(HS4_NAMES)
  return data.sort_values("hs4").reset_index(drop=True)

## Figure 4.6: Heading profiles

One row for each of the 68 HS4 headings sorted by total reconciled value: *(a)* Total F *(b)* Median and IQR of `D` with value-weighted D<sub>w</sub> *(c)* Median `S` with value-weighted S<sub>w</sub> *(d)* Total `G` as share of total `F`.

In [ ]:
def fig_4_6_heading_profiles(con, min_flow=MIN_FLOW, years=STUDY_YEARS, sort_by="F_total", d_xlim=(0, 1), s_xlim=(-0.3, 0.3),
                             ratio_xlim=(0, None), figsize=(11, 12.5), name="fig_4_6_heading_profiles", heading_font_size=7, legend_font_size=7, save=SAVE_OUTPUTS):
  data = heading_stats(con, min_flow, years=STUDY_YEARS).sort_values(sort_by, ascending=True).reset_index(drop=True)
  y = np.arange(len(data))
  colors = [CHAPTER_COLORS[h] for h in data["hs2"]]
  fig, axes = plt.subplots(1, 4, figsize=figsize, sharey=True, gridspec_kw=dict(wspace=0.08))

  # Panel (a): Total F
  ax = axes[0]
  ax.hlines(y, data["F_total"].min() / 3, data["F_total"], color="#DDDDDD", lw=1)
  ax.scatter(data["F_total"], y, c=colors, s=20, zorder=3)
  ax.set_xscale("log")
  ax.xaxis.set_major_formatter(USD_FORMATTER)
  ax.set(xlabel=f"Total F, {years[0]}-{years[1]}\n({value_unit(con)})", title="Trade value")

  # Panel (b): Mean & IQR of D, plus D_w
  ax = axes[1]
  # Line from D_p25 to D_p75
  ax.hlines(y, data["D_p25"], data["D_p75"], color=colors, lw=2.2, alpha=0.45)
  # Plot median D_p50 and D_w
  ax.scatter(data["D_p50"], y, c=colors, s=20, zorder=3)
  ax.scatter(data["D_w"], y, marker="|", c="black", s=45, zorder=4)
  # Create legend manually
  ax.scatter([], [], c=GREY, s=20, label="Median D (bar = IQR)")
  ax.scatter([], [], marker="|", c="black", s=45, label="Value-weighted $D_w$")
  ax.set(xlim=d_xlim, xlabel="D", title="Normalized Discrepancy (D)")
  ax.set_xticks(np.arange(d_xlim[0], d_xlim[1] - 0.01, 0.2))
  ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.030), fontsize=legend_font_size)

  # Panel (c): Median S and S_w
  ax = axes[2]
  # Vertical faint line at x=0
  ax.axvline(0, color=GREY, lw=0.6)
  # Plot median S_p50 and S_w
  ax.scatter(data["S_p50"], y, c=colors, s=20, zorder=3)
  ax.scatter(data["S_w"], y, marker="|", c="black", s=45, zorder=4)
  # Create legend manually
  ax.scatter([], [], c=GREY, s=20, label="Median S")
  ax.scatter([], [], marker="|", c="black", s=45, label="Value-weighted $S_w$")
  ax.set(xlim=s_xlim, xlabel="S", title="Signed Discrepancy (S)")
  ax.set_xticks(np.round(np.arange(s_xlim[0] + 0.1, s_xlim[1] - 0.01, 0.1), 2))
  ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.030), fontsize=legend_font_size)

  # Panel (d): G over F
  ax = axes[3]
  ax.hlines(y, 0, 100 * data["G_over_F"], color="#DDDDDD", lw=1)
  ax.scatter(100 * data["G_over_F"], y, c=colors, s=20, zorder=3)
  ax.set(xlim=ratio_xlim, xlabel="Total G / total F (%)", title="Discrepancy relative to value")

  axes[0].set_yticks(y, [f"{h} {n}" for h, n in zip(data["hs4"], data["name"])], fontsize=heading_font_size)
  for tick, color in zip(axes[0].get_yticklabels(), colors):
    tick.set_color(color)
  for ax in axes:
    ax.set_ylim(-1, len(data))
    ax.grid(axis="y", color="#F2F2F2", lw=0.5)
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, data

## Figure 4.7 and Table 4.2: Grouping headings by discrepancy profile

Ward hierarchical clustering of of the 68 headings on their standardized discrepancy profiles. Number of groups `k` is chosen
by maximizing average silhouette width unless `k` is given.

In [ ]:
CLUSTER_FEATURES = ("D_p10", "D_p25", "D_p50", "D_p75", "D_p90", "D_w", "S_p25", "S_p50", "S_p75",
                    "share_D_gt05", "two_sided_share")


def heading_clusters(stats_df, features=CLUSTER_FEATURES, k=None, k_range=(2, 6)):
  """Ward clustering of headings: returns (labels, linkage matrix, silhouette by k, chosen k).

  stats_df  : dataframe with the full set of HS4-level statistics as returned by `heading_stats()`
  features  : list of columns to use as clustering profile
  k         : number of groups (default: maximum average silhouette over k_range); drawn as a dashed red cut
  k_range   : range of k to try
  """
  # Filter the chosen statistics from the chosen statistics df
  X = stats_df.set_index("hs4")[list(features)]
  # Standardize values
  Z = ((X - X.mean()) / X.std()).values
  # Calculate distance tree using Ward’s minimum variance
  L = linkage(Z, method="ward")
  # Cut the tree for each candidate k and calculate silhouette score
  silhouettes = {kk: silhouette_score(Z, fcluster(L, kk, "maxclust")) for kk in range(k_range[0], k_range[1] + 1)}
  k = k or max(silhouettes, key=silhouettes.get)
  # Cut the tree at chosen k
  raw = pd.Series(fcluster(L, k, "maxclust"), index=X.index)
  # Group the median D of each cluster and sort output
  #  eg. if ranking = [3, 1, 4, 2], raw cluster 3 has lowest D, then 1, 4, 2
  ranking = X["D_p50"].groupby(raw).median().sort_values().index      # group 1 = lowest median D
  # Pair each cluster with their new rank (i.e., {3: 1, 1: 2, 4: 3, 2: 4}) and map to every row in `raw`
  #   label 1 = cluster with lowest median, ..., label k = highest median
  labels = raw.map({old: new for new, old in enumerate(ranking, start=1)})
  return labels, L, silhouettes, k


def fig_4_7_heading_clusters(con, min_flow=MIN_FLOW, k=None, k_range=(2, 6), color_k=None, extra_cuts=(3,),
                          group_names=None, features=CLUSTER_FEATURES, figsize=(8, 12), name="fig_4_7_heading_clusters",
                          save=SAVE_OUTPUTS):
  """Dendrogram of the headings.

  k          : number of groups (defaults to maximum average silhouette over k_range)
  color_k   : color branches by the groups at this k (defaults to k)
  extra_cuts : any additional dotted cuts for reference, e.g. (3,)
  Returns the heading statistics with a `group` column for `color_k` groups (1 = lowest median D).
  """
  data = heading_stats(con, min_flow)
  labels, L, silhouettes, k = heading_clusters(data, features, k, k_range)
  color_k = color_k or k
  color_labels = heading_clusters(data, features, color_k)[0]
  group_names = group_names or {g: f"Group {g}" for g in range(1, color_k + 1)}
  # Helper function to calculate distance threshold on dendrogram to cut the tree and result in kk clusters
  cut_at = lambda kk: (L[-(kk - 1), 2] + L[-kk, 2]) / 2
  set_link_color_palette([FOCUS, ACCENT, "#5FA8A0", "#9B59B6", "#7A9A3A", "#8C510A"])
  fig, ax = plt.subplots(figsize=figsize)
  tree = dendrogram(L, labels=data["hs4"].tolist(), orientation="right", ax=ax,
                    color_threshold=cut_at(color_k), above_threshold_color=GREY, leaf_font_size=7)
  set_link_color_palette(None)
  # Sey y-axis (tree["ivl"] contains original hs4 in the correct order):
  ax.set_yticklabels([f"{h} {HS4_NAMES[h]}" for h in tree["ivl"]])
  leaf_color = dict(zip(tree["ivl"], tree["leaves_color_list"]))
  handles = []
  for g in range(1, color_k + 1):
    members = color_labels[color_labels == g].index
    # Dummy line to manually build legend
    handles.append(mpl.lines.Line2D([], [], color=leaf_color[members[0]], lw=2,
                                    label=f"{g}. {group_names[g]} ({len(members)})"))
  ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(1, 0.96), title="Groups at k = %d" % color_k,
            title_fontsize=7.5, frameon=True, facecolor="white", edgecolor="none", framealpha=1)
  # Color labels by HS2 chapter
  for tick in ax.get_yticklabels():
    tick.set_color(CHAPTER_COLORS[tick.get_text()[:2]])
  # Draw cut at optimal k
  ax.axvline(cut_at(k), color=CONTRAST, ls="--", lw=0.8)
  ax.text(cut_at(k), 1.0, f" k = {k}", transform=ax.get_xaxis_transform(), color=CONTRAST, fontsize=7, va="bottom")
  # Draw extra cuts
  for kk in extra_cuts:
    ax.axvline(cut_at(kk), color=GREY, ls=":", lw=0.8)
    ax.text(cut_at(kk), 1.0, f" k = {kk}", transform=ax.get_xaxis_transform(), color=GREY, fontsize=7, va="bottom")
  ax.set_xlabel("Ward linkage distance (standardised discrepancy profile)")
  for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
  fig.subplots_adjust(left=0.32, right=0.97, top=0.97, bottom=0.05)
  if save:
    save_figure(fig, name)
  return fig, data.assign(group=data["hs4"].map(color_labels)), silhouettes, k


def tab_4_2_product_groups(con, groups, min_flow=MIN_FLOW, years=STUDY_YEARS, group_names=None, n_examples=4,
                             name="tab_4_2_product_groups", save=SAVE_OUTPUTS):
  """Summary per heading group. `groups` is an hs4 series -> group number (from fig_heading_clusters())."""
  group_names = group_names or {g: f"Group {g}" for g in sorted(groups.unique())}
  stats_ = heading_stats(con, min_flow, years)
  rows = []
  for g in sorted(groups.unique()):
    members = groups[groups == g].index.tolist()
    codes = ", ".join(f"'{h}'" for h in members)
    r = query(con, f"""
      SELECT count(*) AS n_flows, median(D) AS D_p50, sum(G) / sum(E + I) AS D_w, median(S) AS S_p50,
             sum(E - I) / sum(E + I) AS S_w, sum(G) / sum(F) AS G_over_F
      FROM panel WHERE {sample_filter(min_flow, years)} AND hs4 IN ({codes})
    """).iloc[0]
    # Filter global stats_ df to current group headings
    s = stats_[stats_["hs4"].isin(members)]
    # Exctract top commodities as examples
    examples = s.sort_values("F_total", ascending=False)["name"].head(n_examples).str.lower().tolist()
    # Write table row
    rows.append({"group": group_names[g], "headings": len(members), "F_share": s["F_total"].sum() / stats_["F_total"].sum(),
                 "two_sided_share": (s["two_sided_share"] * s["n_flows"]).sum() / s["n_flows"].sum(),
                 **r.to_dict(), "examples": ", ".join(examples)})
  data = pd.DataFrame(rows)
  table = pd.DataFrame({
    "Group": data["group"], "Headings": data["headings"],
    "Share of F (%)": data["F_share"].map(lambda v: fmt(v, 1, pct=True)),
    "Flows": data["n_flows"].map(lambda v: f"{v:,.0f}"),
    "Median D": data["D_p50"].map(lambda v: fmt(v, 3)), "D_w": data["D_w"].map(lambda v: fmt(v, 3)),
    "Median S": data["S_p50"].map(lambda v: fmt(v, 3)), "S_w": data["S_w"].map(lambda v: fmt(v, 3)),
    "Largest headings": data["examples"],
  })
  if save:
    write_table(table, name)
  return table, data

## Country-level Statistics

Used by Figures 4.8, 4.12, and Table C.2

In [ ]:
def country_role_stats(con, min_flow=MIN_FLOW, years=STUDY_YEARS, exclude_hs4=(), extra=None):
  """One row per country with discrepancy statistics as exporter (x) and as importer (m)."""
  out = []
  for role, partner, suffix in [("exporter", "importer", "_x"), ("importer", "exporter", "_m")]:
    d = query(con, f"""
      SELECT {role} AS country, count(*) AS n_flows, count(DISTINCT {partner}) AS n_partners, sum(F) AS F,
        median(D) AS D_p50, sum(G) / sum(E + I) AS D_w, sum(E - I) / sum(E + I) AS S_w,
        sum(G) AS G_sum, sum(G) / count(*) AS G_per_flow
      FROM panel WHERE {sample_filter(min_flow, years, exclude_hs4, extra=extra)} GROUP BY {role}
    """).set_index("country")
    out.append(d.add_suffix(suffix))
  # Outer join on exporter and importer stats per country
  data = out[0].join(out[1], how="outer")

  # Overall total F for each country, used for marker size
  total_F = query(con, f"""
    SELECT country, sum(F) AS F_total FROM (
      SELECT exporter AS country, F FROM panel WHERE year BETWEEN {years[0]} AND {years[1]}
      UNION ALL SELECT importer, F FROM panel WHERE year BETWEEN {years[0]} AND {years[1]}
    ) GROUP BY country
  """).set_index("country")
  return data.join(total_F)

## Figure 4.8: Countries as exporters vs importers

*(a)* Value-weighed D<sub>w</sub> of exports against imports by country *(b)* the same for value-weighted S<sub>w</sub>.

Country size in graph proportional to square root of country's total `F`, with labels on the `n_largest` traders and the furthest `n_extreme` countries from the diagonal.

In [ ]:
def fig_4_8_country_roles(con, min_flow=MIN_FLOW, min_flows_per_role=500, n_largest=6, n_extreme=8, d_lim=(0, 0.95),
                        s_lim=(-0.6, 0.5), figsize=(11, 5), name="fig_4_8_country_roles", save=SAVE_OUTPUTS):
  """
  min_flows_per_role  : filter countries with a minimum number of flows in each role
  n_largest           : the top traders that will be labelled on the graph
  n_extreme           : the furthest countries from the diagonal that will be labelled on the graph
  d_lim               : limits for the D axes
  s_lim               : limits for the S axes
  """
  data = country_role_stats(con, min_flow)
  data = data[(data["n_flows_x"] >= min_flows_per_role) & (data["n_flows_m"] >= min_flows_per_role)].copy()
  # Calculate marker size within a min and max bound
  size = np.clip(np.sqrt(data["F_total"] / 1e9) * 4, 6, 250)
  fig, axes = plt.subplots(1, 2, figsize=figsize)

  ax = axes[0]
  ax.scatter(data["D_w_x"], data["D_w_m"], s=size, color=FOCUS, alpha=0.5, edgecolor="white", lw=0.4)
  ax.plot(d_lim, d_lim, color=GREY, lw=0.8, ls="--")
  ax.set(xlim=d_lim, ylim=d_lim, xlabel="Value-weighted D as exporter", ylabel="Value-weighted D as importer")
  gap = (data["D_w_x"] - data["D_w_m"]).abs()
  chosen = list(dict.fromkeys(data["F_total"].nlargest(n_largest).index.tolist() + gap.nlargest(n_extreme).index.tolist()))
  label_points(ax, data.loc[chosen, "D_w_x"].values, data.loc[chosen, "D_w_m"].values, chosen)
  panel_label(ax, "a")

  ax = axes[1]
  ax.scatter(data["S_w_x"], data["S_w_m"], s=size, color=FOCUS, alpha=0.5, edgecolor="white", lw=0.4)
  ax.plot(s_lim, s_lim, color=GREY, lw=0.8, ls="--")
  # Add lines crossing at (0,0)
  ax.axhline(0, color=GREY, lw=0.5)
  ax.axvline(0, color=GREY, lw=0.5)
  ax.set(xlim=s_lim, ylim=s_lim, xlabel="Value-weighted S as exporter", ylabel="Value-weighted S as importer")
  gap = (data["S_w_x"] - data["S_w_m"]).abs()
  chosen = list(dict.fromkeys(data["F_total"].nlargest(n_largest).index.tolist() + gap.nlargest(n_extreme).index.tolist()))
  label_points(ax, data.loc[chosen, "S_w_x"].values, data.loc[chosen, "S_w_m"].values, chosen)
  panel_label(ax, "b")

  # Add text to clarify figure
  ax.text(0.02, 0.97, "under-reports in both roles", transform=ax.transAxes, va="top", fontsize=7, color=GREY)
  ax.text(0.75, 0.03, "over-reports in both roles", transform=ax.transAxes, va="bottom", fontsize=7, color=GREY)

  fig.tight_layout()

  if save:
    save_figure(fig, name)
  return fig, data

## Figure 4.9: Specific Country-HS4 discrepancies

Value-weighted S<sub>w</sub> of the largest `top_n` exporters for each HS4 heading provided in `hs4_codes` (max 8, i.e. panels *(a)* through *(h)*). Also draws a dashed line for the headings' pooled value-weighted S<sub>w</sub> across all exporters (representing the common offset for the commodity). Countries below the threshold `highlight_below` are colored.

In [ ]:
def fig_4_9_wood_exporters(con, hs4_codes=("4403", "4407"), top_n=25, min_flow=MIN_FLOW, highlight_below=-0.25,
                           xlim=(-0.8, 0.8), figsize=(11, 7.2), name="fig_4_9_wood_exporters", save=SAVE_OUTPUTS):
  """
  hs4_codes       : each HS4 code that will receive its own S and D panels (max 8)
  top_n           : the amount of top exporters to show on each panel
  highlight_below : highlight countries with S_w < this value
  xlim            : upper and lower bounds for the S axis
  """
  fig, axes = plt.subplots(1, len(hs4_codes), figsize=figsize, gridspec_kw=dict(wspace=0.35))
  axes = np.atleast_1d(axes)
  out = {}
  for ax, hs4, letter in zip(axes, hs4_codes, "abcdefgh"):
    # Calculate statistics using sample filter and total F across the dataset (for ordering an d labeling)
    d = query(con, f"""
      WITH s AS (
        SELECT exporter, count(*) AS n_flows, sum(E) AS E, sum(I) AS I, sum(E - I) / sum(E + I) AS S_w
        FROM panel WHERE {sample_filter(min_flow)} AND hs4 = '{hs4}' GROUP BY exporter
      ), t AS (
        SELECT exporter, sum(F) AS F_total FROM panel
        WHERE hs4 = '{hs4}' AND year BETWEEN {STUDY_YEARS[0]} AND {STUDY_YEARS[1]} GROUP BY exporter
      )
      SELECT * FROM s JOIN t USING (exporter) ORDER BY F_total DESC LIMIT {top_n}
    """).iloc[::-1].reset_index(drop=True)
    # Calculate overall Sw for all exporters in sample
    overall = con.execute(f"SELECT sum(E - I) / sum(E + I) FROM panel WHERE {sample_filter(min_flow)} AND hs4 = '{hs4}'").fetchone()[0]
    y = np.arange(len(d))
    colors = [CONTRAST if s < highlight_below else GREY for s in d["S_w"]]
    # Line at Sw=0
    ax.axvline(0, color="black", lw=0.6)
    # Dashed line at Sw = overall average with own lable
    ax.axvline(overall, color=FOCUS, lw=1, ls="--")
    ax.text(overall, len(d) - 0.2, f" all exporters ({overall:+.2f})", color=FOCUS, fontsize=7, va="bottom")
    # All h-lines from 0 to Sw, colored relative to `highlight_below`
    ax.hlines(y, 0, d["S_w"], color=colors, lw=2.2)
    ax.plot(d["S_w"], y, "o", color="black", ms=3)
    # Label each country as {c}  {usd(f)}, e.g. USA 3.1M
    ax.set_yticks(y, [f"{c}  {usd(f)}" for c, f in zip(d["exporter"], d["F_total"])], fontsize=7.5)
    ax.set(xlim=xlim, ylim=(-1, len(d) + 0.8), xlabel="Value-weighted S of the exporter's two-sided flows")
    # Label panel
    ax.set_title(f"{hs4} {HS4_NAMES[hs4]}")
    panel_label(ax, letter, x=-0.2)
    out[hs4] = d.assign(S_w_all_exporters=overall)

  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, out

## Figure 4.10: Concentration of absolute discrepancy

*(a)* The aggregated share of total `G` across the study period for all headings and country pairs in the sample, ranked by largest aggregated `G` *(b)* The share of each year's total G coming from the flows in the specified top percentiles `top_shares`, when ranked by `G`.

In [ ]:
def fig_4_10_dyad_concentration(con, min_flow=MIN_FLOW, years=STUDY_YEARS, top_shares=(0.01, 0.10),
                                marks=(0.5, 0.8), figsize=(11, 4), name="fig_4_10_dyad_concentration", save=SAVE_OUTPUTS):
  """
  top_shares  : the top percentiles for which share of annual total G is calculated (max 3)
  marks       : the shares of total G at which to draw horizontal lines and calculate number of pairs
  """
  where = sample_filter(min_flow, years)
  dyads = query(con, f"""
    SELECT exporter, importer, sum(G) AS G, sum(F) AS F, sum(E) AS E, sum(I) AS I, count(*) AS n_flows
    FROM panel WHERE {where} GROUP BY exporter, importer ORDER BY G DESC
  """)
  # Calculate cumulative share of G for all `dyads` ordered by G (descending)
  dyads["cum_share"] = dyads["G"].cumsum() / dyads["G"].sum()

  # Filter df to include only flows in each desired top_share share of flows as top_X
  shares_sql = ", ".join(f"sum(G) FILTER (WHERE rank_ < {s}) / sum(G) AS top_{round(s * 100)}" for s in top_shares)
  # Largest flow gets rank_ = 0 and smallest flow rank_ = 1 per year
  annual = query(con, f"""
    SELECT year, {shares_sql}
    FROM (SELECT year, G, percent_rank() OVER (PARTITION BY year ORDER BY G DESC) AS rank_ FROM panel WHERE {where})
    GROUP BY year ORDER BY year
  """).set_index("year")

  fig, axes = plt.subplots(1, 2, figsize=figsize)
  # Panel (a)
  ax = axes[0]
  # Plot cumulative share of G against nr. of country pairs on the x-axis
  ax.plot(np.arange(1, len(dyads) + 1), 100 * dyads["cum_share"], color=CONTRAST, lw=2)
  # Mark h-line for each desired cutoff
  for m in marks:
    # Calculate nr. of pair below cutoff, draw, and label
    n_m = int((dyads["cum_share"] < m).sum() + 1)
    ax.axhline(100 * m, color=GREY, lw=0.6, ls="--")
    ax.annotate(f"{n_m:,} pairs", xy=(n_m, 100 * m), xytext=(6, -10), textcoords="offset points", fontsize=7)
  ax.set_xscale("log")
  ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
  ax.set(ylim=(0, 100), xlabel="Country pairs ranked by cumulative G (log)", ylabel="Cumulative share of total G (%)")
  panel_label(ax, "a")

  ax = axes[1]
  # Add more colors to expand beyond limit of 3 top_shares
  colors = [CONTRAST, ACCENT, FOCUS]
  items = []
  for s, color in zip(top_shares, colors):
    # Filter the ranked 'annual' df for each top share desired
    col = f"top_{round(s * 100)}"
    # Plot the top_X column against each year
    ax.plot(annual.index, 100 * annual[col], color=color, lw=2)
    # Make label and store point and color for layout
    items.append((f"Top {s:.0%} of flows", 100 * annual[col].iloc[-1], color))
  ax.set(xlim=years, ylim=(0, 100), xlabel="Year", ylabel="Share of annual total G (%)")
  end_labels(ax, items, years[1] + 0.4)
  panel_label(ax, "b")
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, dyads, annual

## Discrepancy persistence

For every `exporter x importer x heading` relationship:
* `H = 1` in a year if `D` is above the `percentile` of D for that heading and year
* `T` = the number of two-sided years
* `pi` = share of two-sided years with `H = 1`
* `longest_run` = the longest streak of consecutive years with `H = 1`

The expected share of observations in the top p percentile is the mean of `binom.sf(⌈0.5T⌉−1, T, p)` over relationships.

In [ ]:
def persistence_table(con, min_flow=MIN_FLOW, percentile=0.9, years=STUDY_YEARS, exclude_hs4=(), extra=None):
  where = sample_filter(min_flow, years, exclude_hs4, extra=extra)
  return query(con, f"""
    WITH flows AS (SELECT * FROM panel WHERE {where}),
    -- Calculate threshold based on given percentile for each heading
    thresholds AS (SELECT hs4, year, quantile_cont(D, {percentile}) AS d_cut FROM flows GROUP BY hs4, year),

    -- Mark flows at or beyond the threshold for their given hs4 and year (H = 0 or H = 1)
    marked AS (
      SELECT f.exporter, f.importer, f.hs4, f.year, f.D, f.E, f.I, f.F, f.G, (f.D >= t.d_cut)::INT AS H
      FROM flows f JOIN thresholds t USING (hs4, year)
    ),

    -- Calculate streaks of consecutive years with H = 1
    --  (by grouping using `year - row_number()`, consecutive streaks are placed together)
    streaks AS (
      SELECT exporter, importer, hs4, count(*) AS streak
      FROM (SELECT exporter, importer, hs4,
                   year - row_number() OVER (PARTITION BY exporter, importer, hs4 ORDER BY year) AS block
            FROM marked WHERE H = 1)
      GROUP BY exporter, importer, hs4, block
    ),

    -- Mark the longest run of consecutive H = 1 for each pair x heading relationship
    longest AS (SELECT exporter, importer, hs4, max(streak) AS longest_run FROM streaks GROUP BY ALL)
    SELECT m.exporter, m.importer, m.hs4, count(*) AS T, sum(H) AS H_sum, sum(H) / count(*) AS pi,
      coalesce(any_value(l.longest_run), 0) AS longest_run, min(year) AS first_year, max(year) AS last_year,
      median(D) AS D_p50, sum(E) AS E_sum, sum(I) AS I_sum, sum(F) AS F_sum, sum(G) AS G_sum,
      sum(E - I) / sum(E + I) AS S_w
    FROM marked m LEFT JOIN longest l USING (exporter, importer, hs4)
    GROUP BY m.exporter, m.importer, m.hs4
  """)

def longest_run_pmf(T, p):
  """Exact distribution of the longest run of successes (with rate `p`) in `T` Bernoulli trials.
  Example run for T=3, p=0.5, k=2:
  - initial state = [1, 0, 0]
  - Loop 3 times:
  -   Loop 1: new[0] = state.sum()*(1-p) = 1.0*0.5 = 0.5
              new[1:] = state[:-1]*p = [1, 0]*0.5 = [0.5, 0]
              state = [0.5, 0.5, 0]
  -   Loop 2: new[0] = state.sum()*(1-p) = 1.0*0.5 = 0.5
              new[1:] = state[:-1]*p = [0.5, 0.5]*0.5 = [0.25, 0.25]
              state = [0.5, 0.25, 0.25]
  -   Loop 3: new[0] = state.sum()*(1-p) = 1.0*0.5 = 0.5
              new[1:] = state[:-1]*p = [0.5, 0.25]*0.5 = [0.25, 0.125]
              state = [0.5, 0.25, 0.125]
  - state.sum() = 0.5 + 0.25 + 0.125 = 0.875 => P(longest run ≤ 2 after 3 trials with p=0.5)
  - cdf[2] = 0.875 => P(longest run ≤ 2 after 3 trials with p=0.5)
  """
  # List to store P(longest run ≤ k) with k = 0,1,...,T
  cdf = []
  # Calculate probability longest run is below k after T trials (tracking current run length as a Markov chain)
  for k in range(T + 1):
    # State vector of length k+1 (for states 0, 1, ..., k), starting at state 0 with probability 1
    # - state[j] = the probability that the current consecutive run of successes has length j
    state = np.zeros(k + 1)
    state[0] = 1.0
    # T Bernoulli trials
    for _ in range(T):
      # New state vector
      new = np.zeros(k + 1)
      # Failure, with probability (1-p), resets run length to 0
      new[0] = state.sum() * (1 - p)
      # Success, with probability (p), increeases run length by 1
      new[1:] = state[:-1] * p
      state = new
    # After T trials, state.sum() is the total probability P(longest run ≤ k)
    cdf.append(state.sum())
  # Convert CDF to PMF
  return np.diff(np.r_[0.0, cdf])

def persistence_null(pers, p, pi_bins):
  """Expected distribution of pi and longest run, if H were randomly distributed across years (rate p).
  pers    : DataFrame with a "T" column, one row per series as returned by `persistence_table()`
  pi_bins : histogram bins (e.g., np.linspace(0, 1, 21))
  """
  # pi under null hypothesis for each bin
  pi_null = np.zeros(len(pi_bins) - 1)
  # Maximum possible longest run length (max T + 1)
  run_null = np.zeros(STUDY_YEARS[1]-STUDY_YEARS[0]+1+1)
  # Group series by their length T (e.g. 500 series have T=20 => T=20, count=500)
  for T, count in pers["T"].value_counts().items():
    # All the possible hits for series of length T (e.g. with T=20 => h = [0 1 ... 20])
    h = np.arange(T + 1)
    # Under null hypothesis, total H ~ Binomial(T, p). For every h, calculate:
    # - h / T = pi
    # - stats.binom.pmf(h, T, p) = probability of that h
    # - histogram(pi*stats.binom.pmf(h, T, p)) = P(pi falls in bin) for a series of length T
    # - P(pi falls in bin, T) * count = weight by how many series have this T
    pi_null += count * np.histogram(h / T, bins=pi_bins, weights=stats.binom.pmf(h, T, p))[0]
    # Same for the longest run, using longest_run_pmf(T, p), adding it to the first T slots of run_null
    run_null[: T + 1] += count * longest_run_pmf(T, p)
  # Normalize
  return pi_null / len(pers), run_null / len(pers)

## Figure 4.11: Persistence of discrepancies in top decile

Relationships with at least `min_years` of two-sided reporting years. Draws red lines for the expected distribution if years occurred in the top percentile by chance (binomial null against observed rate `p`).

In [ ]:
def fig_4_11_persistence(con, min_flow=MIN_FLOW, percentile=0.9, min_years=10, pi_threshold=0.5, n_size_groups=10,
                         figsize=(8, 4), name="fig_4_11_persistence", save=SAVE_OUTPUTS):
  pers_all = persistence_table(con, min_flow, percentile)
  # Observed pi
  p = pers_all["H_sum"].sum() / pers_all["T"].sum()
  # Filter min T = min_years
  pers = pers_all[pers_all["T"] >= min_years].copy()
  # Bins for pi calculations
  bins = np.linspace(0, 1, 21)
  # Get null distributions
  pi_null, run_null = persistence_null(pers, p, bins)
  # Get (normalized) observed distributions
  pi_obs = np.histogram(pers["pi"], bins=bins)[0] / len(pers)
  run_obs = np.bincount(pers["longest_run"].astype(int), minlength=STUDY_YEARS[1]-STUDY_YEARS[0]+2)[:STUDY_YEARS[1]-STUDY_YEARS[0]+2] / len(pers)

  # Share of observations above given threshold, observed and by chance
  share_obs = (pers["pi"] >= pi_threshold).mean()
  share_null = sum(c * stats.binom.sf(np.ceil(pi_threshold * T) - 1, T, p)
                   for T, c in pers["T"].value_counts().items()) / len(pers)

  fig, axes = plt.subplots(1, 2, figsize=figsize)
  # Plot (a)
  centres = (bins[:-1] + bins[1:]) / 2
  ax = axes[0]
  # Plot observed pi values
  ax.step(centres, np.where(pi_obs > 0, 100 * pi_obs, np.nan), where="mid", color=FOCUS, lw=2, label="Observed")
  # Plot null hypothesis values
  ax.plot(centres, np.where(pi_null > 1e-7, 100 * pi_null, np.nan), "o-", color=CONTRAST, ms=3, lw=1.2,
          label=f"Independent years (p = {p:.2f})")
  ax.set_yscale("log")
  ax.set(ylim=(1e-3, 100), xlabel=r"Persistence $\pi$ (share of years in the top decile)", ylabel="Share of relationships (%)")
  ax.legend(loc="upper right")
  panel_label(ax, "a")

  # Plot (b)
  ax = axes[1]
  runs = np.arange(STUDY_YEARS[1]-STUDY_YEARS[0]+2)
  # Plot observed relationships with each possible maximum longest runs
  ax.step(runs, np.where(run_obs > 0, 100 * run_obs, np.nan), where="mid", color=FOCUS, lw=2, label="Observed")
  # Plot null hypothesis values
  ax.plot(runs, np.where(run_null > 1e-7, 100 * run_null, np.nan), "o-", color=CONTRAST, ms=3, lw=1.2,
          label="Independent years")
  ax.set_yscale("log")
  ax.set(xlim=(-0.5, 25), ylim=(1e-3, 100), xlabel="Longest run of consecutive top-decile years", ylabel="Share of relationships (%)")
  ax.legend(loc="upper right")
  panel_label(ax, "b")

  fig.tight_layout()
  if save:
    save_figure(fig, name)

  summary = {"p": p, "n_relationships": len(pers), "share_obs": share_obs, "share_null": share_null,
             "share_run5_obs": (pers["longest_run"] >= 5).mean(), "share_run5_null": run_null[5:].sum()}
  return fig, summary

## Table 4.3: Top persistent high-discrepancy relationships

Top `top_n` persistent high-discrepancy relationships with minimum pi `min_pi` and minimum `T` of `min_years`, ranked by `sort_by`

In [ ]:
def tab_4_3_top_persistent(con, min_flow=MIN_FLOW, min_years=10, min_pi=0.5, top_n=15, sort_by="G_sum",
                             exclude_hs4=("4907",), name="tab_4_3_top_persistent", save=SAVE_OUTPUTS):
  pers = persistence_table(con, min_flow)
  top = (pers[(pers["T"] >= min_years) & (pers["pi"] >= min_pi) & ~pers["hs4"].isin(exclude_hs4)]
         .sort_values(sort_by, ascending=False).head(top_n).reset_index(drop=True))

  table = pd.DataFrame({
    "Exporter $\\rightarrow$ importer": top["exporter"] + " $\\rightarrow$ " + top["importer"],
    "Heading": top["hs4"] + " " + top["hs4"].map(HS4_NAMES),
    "Years": top["first_year"].astype(str) + "--" + top["last_year"].astype(str),
    "$T$": top["T"], "$\\pi$": top["pi"].map(lambda v: fmt(v, 2)), "Run": top["longest_run"],
    "$\\Sigma E$ (M USD)": (top["E_sum"] / 1e6).map(lambda v: fmt(v, 0)),
    "$\\Sigma I$ (M USD)": (top["I_sum"] / 1e6).map(lambda v: fmt(v, 0)),
    "$S_w$": top["S_w"].map(lambda v: fmt(v, 2)),
  })

  if save:
    write_table(table, name)
  return table, top

## Figure 4.12 and Table 4.4: Basic measures of the discrepancy network

Countries are nodes, and edges `x -> m` represent a pair that has had at least one two-sided flow in the sample in a year (of any heading). (a) network density over the years (b) number of export partners versus the value-weighted D<sub>w</sub> of the node's exports (c) mean G per export flow versus import flow

In [ ]:
def annual_network(con, min_flow=MIN_FLOW, years=STUDY_YEARS, two_sided=True):
  """Nodes, directed edges, density, and reciprocity of the annual network of country pairs."""
  where = sample_filter(min_flow if two_sided else 0, years, two_sided=two_sided)
  if not two_sided:
    where += " AND reporting_case <> 'neither'"
  return query(con, f"""
    -- Get all edges in the sample
    WITH edges AS (SELECT DISTINCT year, exporter, importer FROM panel WHERE {where}),

    -- Get the nodes per year
    nodes AS (SELECT year, count(DISTINCT c) AS nodes FROM
              (SELECT year, exporter AS c FROM edges UNION SELECT year, importer FROM edges) GROUP BY year),

    -- Get reciprocated edges per year
    recip AS (SELECT e.year, count(*) AS edges, sum((r.exporter IS NOT NULL)::INT) AS reciprocated
              FROM edges e LEFT JOIN edges r ON r.year = e.year AND r.exporter = e.importer AND r.importer = e.exporter
              GROUP BY e.year)

    -- Get network statistics
    SELECT year, nodes, edges, edges / (nodes * (nodes - 1.0)) AS density, reciprocated / edges AS reciprocity,
           edges / nodes AS mean_out_degree
    FROM recip JOIN nodes USING (year) ORDER BY year
  """).set_index("year")


def fig_4_12_network_basics(con, min_flow=MIN_FLOW, years=STUDY_YEARS, density_years=(1995, 2023), min_flows_per_role=200,
                            n_label=15, figsize=(13, 4.2), name="fig_4_12_network_basics", save=SAVE_OUTPUTS):
  """2024 is left out by default because of the reporting lag."""
  net_two = annual_network(con, min_flow, density_years, two_sided=True)
  net_all = annual_network(con, min_flow, density_years, two_sided=False)
  roles = country_role_stats(con, min_flow, years)
  fig, axes = plt.subplots(1, 3, figsize=figsize)

  # Panel (a)
  ax = axes[0]
  ax.plot(net_all.index, 100 * net_all["density"], color=GREY, lw=2)
  ax.plot(net_two.index, 100 * net_two["density"], color=FOCUS, lw=2)
  ax.set(xlim=density_years, ylim=(0, None), xlabel="Year", ylabel="Density (% of possible directed pairs)")
  end_labels(ax, [("All reported flows", 100 * net_all["density"].iloc[-1], GREY),
                  (f"Two-sided, F $\\geq$ {usd(min_flow)}", 100 * net_two["density"].iloc[-1], FOCUS)], density_years[1] + 0.4)
  panel_label(ax, "a")

  # Panel (b)
  ax = axes[1]
  # Filter countries with minimum flows
  d = roles[roles["n_flows_x"] >= min_flows_per_role]
  # Get correlation coefficient
  r = np.corrcoef(np.log(d["n_partners_x"]), d["D_w_x"])[0, 1]
  ax.scatter(d["n_partners_x"], d["D_w_x"], s=np.clip(np.sqrt(d["F_total"] / 1e9) * 4, 5, 200),
             color=ACCENT, alpha=0.55, edgecolor="white", lw=0.3)
  ax.set_xscale("log")
  # Manually formatting ticks for better visualization
  ax.xaxis.set_major_locator(mpl.ticker.FixedLocator([5, 10, 20, 50, 100, 200]))
  ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
  ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
  ax.set(ylim=(0, 1), xlabel="Number of importing partners (1995-2024)", ylabel="Value-weighted D of exports")
  ax.text(0.98, 0.97, f"r(log partners, D) = {r:.2f}", transform=ax.transAxes, ha="right", va="top", fontsize=7)
  # Add labels to nodes with the largest D_w
  chosen = d["D_w_x"].nlargest(n_label).index.tolist()
  label_points(ax, d.loc[chosen, "n_partners_x"].values, d.loc[chosen, "D_w_x"].values, chosen)
  panel_label(ax, "b")

  # Panel(c)
  ax = axes[2]
  # Filter countries with minimum flows in both roles
  d = roles[(roles["n_flows_x"] >= min_flows_per_role) & (roles["n_flows_m"] >= min_flows_per_role)]
  ax.scatter(d["G_per_flow_x"], d["G_per_flow_m"], s=np.clip(np.sqrt(d["F_total"] / 1e9) * 4, 5, 200),
             color=FOCUS, alpha=0.5, edgecolor="white", lw=0.3)
  lo = min(d["G_per_flow_x"].min(), d["G_per_flow_m"].min()) / 1.5
  hi = max(d["G_per_flow_x"].max(), d["G_per_flow_m"].max()) * 1.5
  ax.plot([lo, hi], [lo, hi], color=GREY, lw=0.8, ls="--")
  ratio = np.log(d["G_per_flow_x"] / d["G_per_flow_m"])
  chosen = list(dict.fromkeys(d["G_per_flow_x"].nlargest(n_label).index.tolist() + ratio.nlargest(3).index.tolist()))
  ax.set(xscale="log", yscale="log", xlim=(lo, hi), ylim=(lo, hi))
  label_points(ax, d.loc[chosen, "G_per_flow_x"].values, d.loc[chosen, "G_per_flow_m"].values, chosen)
  ax.xaxis.set_major_formatter(USD_FORMATTER)
  ax.yaxis.set_major_formatter(USD_FORMATTER)
  ax.set(xlabel=f"Mean G per export flow ({value_unit(con)})", ylabel=f"Mean G per import flow ({value_unit(con)})")
  panel_label(ax, "c")

  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, net_two, net_all, roles, r


def tab_4_4_network_summary(con, min_flow=MIN_FLOW, table_years=(1995, 2000, 2005, 2010, 2015, 2020, 2023),
                              percentile=0.9, name="tab_4_4_network_summary", save=SAVE_OUTPUTS):
  data = annual_network(con, min_flow, two_sided=True).loc[list(table_years)]
  table = pd.DataFrame({
    "Year": data.index.astype(str), "Countries": data["nodes"], "Directed pairs": data["edges"].map(lambda v: f"{v:,}"),
    "Density (%)": data["density"].map(lambda v: fmt(v, 1, pct=True)),
    "Reciprocity (%)": data["reciprocity"].map(lambda v: fmt(v, 1, pct=True)),
    "Mean degree": data["mean_out_degree"].map(lambda v: fmt(v, 1))
  })
  if save:
    write_table(table, name)
  return table, data

## Table 4.5: Robustness of the main statistics

Key statistics for: (1) all two-sided flows, (2) the analysis sample, and (3) the analysis sample without the flagged headings in `FLAGGED_HS4` and flows in `FLAGGED_FLOWS`.

In [ ]:
def tab_4_5_robustness(con, min_flow=MIN_FLOW, percentile=0.9, min_years=10, pi_threshold=0.5,
                         name="tab_4_5_robustness", save=SAVE_OUTPUTS):
  samples = {
    "All two-sided flows": dict(min_flow=0),
    f"Analysis sample (F $\\geq$ {usd(min_flow)})": dict(min_flow=min_flow),
    "Analysis sample without artefacts": dict(min_flow=min_flow, exclude_hs4=FLAGGED_HS4, extra=FLAGGED_FLOWS),
  }
  # Variance of D tested against these factors
  factors = {"Year": "year", "HS4 heading": "hs4", "Country pair x heading": "exporter, importer, hs4"}
  rows = {}
  for label, kw in samples.items():
    where = sample_filter(kw["min_flow"], exclude_hs4=kw.get("exclude_hs4", ()), extra=kw.get("extra"))
    # Calculate statistics
    s = query(con, f"""
      SELECT count(*) AS n, sum(F) AS F, median(D) AS D_p50, sum(G) / sum(E + I) AS D_w, median(S) AS S_p50,
             sum(E - I) / sum(E + I) AS S_w, sum(G) / sum(F) AS G_over_F
      FROM panel WHERE {where}
    """).iloc[0]
    # Calculate explained variance and persistence
    ve = variance_explained(con, "D", factors, kw["min_flow"], exclude_hs4=kw.get("exclude_hs4", ()),
                            extra=kw.get("extra")).set_index("factor")["epsilon2"]
    pers = persistence_table(con, kw["min_flow"], percentile, exclude_hs4=kw.get("exclude_hs4", ()), extra=kw.get("extra"))
    # p = pers["H_sum"].sum() / pers["T"].sum()
    pers = pers[pers["T"] >= min_years]
    rows[label] = {
      "Two-sided flows": f"{s['n']:,.0f}", "Reconciled value (B USD)": fmt(s["F"] / 1e9, 0),
      "Median D": fmt(s["D_p50"], 3), "Value-weighted $D_w$": fmt(s["D_w"], 3),
      "Median S": fmt(s["S_p50"], 3), "Value-weighted $S_w$": fmt(s["S_w"], 3),
      "Total G / total F (%)": fmt(s["G_over_F"], 1, pct=True),
      "$\\epsilon^2$ year (%)": fmt(ve["Year"], 2, pct=True),
      "$\\epsilon^2$ HS4 heading (%)": fmt(ve["HS4 heading"], 2, pct=True),
      "$\\epsilon^2$ country pair $\\times$ heading (%)": fmt(ve["Country pair x heading"], 1, pct=True),
      f"Relationships with $\\pi \\geq {pi_threshold}$ (%)": fmt((pers["pi"] >= pi_threshold).mean(), 2, pct=True)
    }
  data = pd.DataFrame(rows)
  table = data.reset_index().rename(columns={"index": "Statistic"})
  if save:
    write_table(table, name)
  return table, data

## Figure 5.1: External validity (I)

Discrepancies compared to WWF estimates of illegal logging published in Hirschberger (2008, Annex B). When an estimate is given as a range, the midpoint is calculated. Only estimates of illegal *harvest* are compared, and not estimates of illegal wood *through imports*, since they measure a different concept and are excluded.

Estimates have been manually transcribed to `WWF_2008_ILLEGAL_LOGGING`. Future code revisions will extrapulate the data automatically from a provided csv file.

In [ ]:
WWF_2008_ILLEGAL_LOGGING = pd.DataFrame([
  ("RUS", 27, 50, "harvest"),
  ("TJK", 20, 30, "harvest"),
  ("BIH", 80, 80, "harvest"),
  ("MKD", 25, 30, "harvest"),
  ("ALB", 81, 81, "harvest"),
  ("EST", 50, 50, "harvest"),
  ("LVA", 20, 20, "harvest"),
  ("BGR", 40, 40, "harvest"),
  ("SVK", 10, 10, "harvest"),
  ("CMR", 50, 50, "harvest"),
  ("GAB", 70, 70, "harvest"),
  ("GHA", 60, 60, "harvest"),
  ("NGA", 90, 90, "harvest"),
  ("GNQ", 50, 50, "harvest"),
  ("BEN", 80, 90, "harvest"),
  ("MOZ", 50, 50, "harvest"),
  ("MMR", 50, 50, "harvest"),
  ("IDN", 73, 73, "harvest"),
  ("KHM", 94, 94, "harvest"),
  ("LAO", 45, 45, "harvest"),
  ("PNG", 70, 70, "harvest"),
  ("PHL", 46, 46, "harvest"),
  ("KOR", 30, 30, "harvest"),
  ("BRA", 47, 47, "harvest"),
  ("BOL", 80, 90, "harvest"),
  ("COL", 42, 42, "harvest"),
  ("PER", 80, 80, "harvest"),
  ("ECU", 70, 70, "harvest"),
  ("MEX", 70, 70, "harvest"),
  ("NIC", 50, 50, "harvest"),
  ("HND", 75, 85, "harvest"),

  # Not considered:
  ("CHN", 32, 32, "through imports"),
  ("MYS", 11.8, 11.8, "through imports"),
  ("TWN", 45, 45, "through imports"),
  ("THA", 40, 40, "through imports"),
  ("VNM", 22, 39, "through imports"),
], columns=["iso3", "share_low", "share_high", "basis"])


def external_validity_data(con, estimates=None, hs_sql_cond="hs2 = '44'", years=(2003, 2008), min_flow=MIN_FLOW,
                           min_flows=30, basis="harvest"):
  est = (WWF_2008_ILLEGAL_LOGGING if estimates is None else estimates).copy()
  est["share_mid"] = (est["share_low"] + est["share_high"]) / 2
  where = f"{sample_filter(min_flow, years)} AND ({hs_sql_cond})"
  exports = query(con, f"""SELECT exporter AS iso3, count(*) AS n_flows, sum(G) / sum(E + I) AS D_w, sum(E - I) / sum(E + I) AS S_w
                       FROM panel WHERE {where} GROUP BY exporter HAVING count(*) >= {min_flows}""")
  return est[est["basis"] == basis].merge(exports, on="iso3")


def fig_5_1_external_validity(con, estimates=None, hs_sql_cond="hs2 = '44'", years=(2003, 2008), min_flows=30,
                          figsize=(11, 4.4), name="fig_5_1_external_validity", save=SAVE_OUTPUTS):
  data = external_validity_data(con, estimates, hs_sql_cond, years, min_flows=min_flows)
  fig, axes = plt.subplots(1, 2, figsize=figsize)
  for ax, col, lab, letter in [(axes[0], "S_w", "Value-weighted S of exports ($S_w$)", "a"),
                               (axes[1], "D_w", "Value-weighted D of exports ($D_w$)", "b")]:
    # Correlation and significance for each axes
    rho, p = stats.spearmanr(data["share_mid"], data[col])
    # Error bar from low to high
    ax.errorbar(data["share_mid"], data[col], xerr=[data["share_mid"] - data["share_low"], data["share_high"] - data["share_mid"]],
                fmt="o", color=FOCUS, ecolor="#BBBBBB", ms=5, lw=0.8)
    # Grey like at S_w = 0 for weighted graph
    if col == "S_w":
      ax.axhline(0, color=GREY, lw=0.6)
    ax.set(xlabel="Estimated share of illegal logging (%), WWF (2008)", ylabel=lab)
    ax.set_title(f"Spearman $\\rho$ = {rho:.2f} (p = {p:.3f}, n = {len(data)})", fontsize=8, loc="right")
    ax.figure.canvas.draw()
    label_points(ax, data["share_mid"].values, data[col].values, data["iso3"].tolist(), fontsize=6.5, min_gap=0.036, dx=0.012)
    panel_label(ax, letter)
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, data

## Figure 5.2: External validity (II)
Comparison of calculated discrepancies with:
1. The list of exports and imports in physical units (m^3, kg), as reported by customs agencies of 14 countries for logs, sawnwood, veneer and plywood from 2000 to 2006; published by Chen (2008, Annexes 1-4). If HGL value-based S<sub>w</sub> is in line with Chen's volume based `S`, it can be said that discrepancies are not just a consequence of valuation (e.g. CIF to FOB conversions) or of the HGL reconciliation methodology
2. Partner-recorded imports of logs and sawnwood after export bans where legal exports should stop, meaning continuing trades are not declared by the exporter

In [ ]:
CHEN_2008_PATH = Path(f"{DB_DIR}/chen2008_annex_volumes.csv")   # transcribed from Chen (2008, Annexes 1-4)

EXPORT_BANS = pd.DataFrame([
  ("IDN", ("4403",), 2001, "Indonesia, logs (Oct 2001)", "Chen (2008)"),
  ("IDN", ("4407",), 2004, "Indonesia, sawnwood (Sep 2004)", "Chen (2008)"),
  # ("MMR", ("4403",), 2014, "Myanmar, logs (Apr 2014)", "EIA (2014)"),
  ("LAO", ("4403", "4407"), 2016, "Laos, logs and sawnwood (May 2016)", "To et al. (2017)"),
], columns=["exporter", "hs4", "ban_year", "label", "source"])

def comparison_data(con, path=CHEN_2008_PATH):
  """Chen (2008) pair-year volumes joined to the HGL panel with: `S_vol` from volumes, `S_val` from HGL values."""

  if not os.path.isfile(path):
    print(f"File for external validity comparison (II) was not found in path [{path}]")
    print("Skipping creation of Figure ")
    return None

  chen = pd.read_csv(path, dtype={"hs4": str})
  con.register("chen_volumes", chen)
  data = query(con, """
    SELECT c.*, p.E AS E_val, p.I AS I_val, p.reporting_case
    FROM chen_volumes c LEFT JOIN panel p USING (hs4, year, exporter, importer)
  """)
  data["S_vol"] = (data["E_qty"] - data["I_qty"]) / (data["E_qty"] + data["I_qty"])
  data["S_val"] = (data["E_val"] - data["I_val"]) / (data["E_val"] + data["I_val"])
  data["two_sided"] = (data["E_qty"] > 0) & (data["I_qty"] > 0) & (data["reporting_case"] == "both")
  return data


def export_ban_data(con, bans=EXPORT_BANS, years=STUDY_YEARS, min_partner_value=10e6):
  """Per ban and year, calculate value declared by the exporter (E) and recorded by all partners (I), over all flows.

  - share_declared is left empty in years where the exporter reports no forest-product exports at all
      or where partners record less than `min_partner_value` (avoid noise).
  """
  out = []
  # Per ban, group by year
  for b in bans.itertuples():
    codes = ", ".join(f"'{h}'" for h in b.hs4)
    d = query(con, f"""
      WITH reporting AS (
        SELECT year, sum(E) > 0 AS exporter_reports FROM panel
        WHERE exporter = '{b.exporter}' AND year BETWEEN {years[0]} AND {years[1]} GROUP BY year)
      SELECT year, sum(E) AS E, sum(I) AS I, any_value(r.exporter_reports) AS exporter_reports
      FROM panel JOIN reporting r USING (year)
      WHERE exporter = '{b.exporter}' AND hs4 IN ({codes}) AND year BETWEEN {years[0]} AND {years[1]}
      GROUP BY year ORDER BY year
    """)
    d["label"], d["ban_year"] = b.label, b.ban_year
    d["years_since_ban"] = d["year"] - b.ban_year
    d["share_declared"] = (d["E"] / d["I"]).where(d["exporter_reports"] & (d["I"] >= min_partner_value))
    out.append(d)
  return pd.concat(out, ignore_index=True)


def fig_5_2_external_validity_cases(con, estimates_path=CHEN_2008_PATH, bans=EXPORT_BANS, window=(-6, 8),
                                panel_b_bans=("IDN", "LAO"), figsize=(8, 4.4), name="fig_5_2_external_validity_cases", save=SAVE_OUTPUTS):
  """(a) Chen volume S vs HGL value S, (b) share declared by the exporter around export bans"""
  chen = comparison_data(con, estimates_path)

  if not comparison_data:
    return None, None, None

  two = chen[chen["two_sided"]]
  ban = export_ban_data(con, bans)
  fig, axes = plt.subplots(1, 2, figsize=figsize)

  # Panel (a)
  ax = axes[0]
  colors = dict(zip(["4403", "4407", "4408", "4412"], [CHAPTER_COLORS.get("44", FOCUS), ACCENT, CONTRAST, FOCUS]))
  names = {"4403": "Logs", "4407": "Sawnwood", "4408": "Veneer", "4412": "Plywood"}
  for h, d in two.groupby("hs4"):
    # For each hs4, scatter S_vol vs S_val
    ax.scatter(d["S_vol"], d["S_val"], s=6, alpha=0.35, color=colors[h], lw=0, label=names[h])
  ax.plot([-1, 1], [-1, 1], color=GREY, lw=0.8, ls="--")
  # Calculate overall correlation
  rho = stats.spearmanr(two["S_vol"], two["S_val"]).statistic
  ax.set(xlim=(-1, 1), ylim=(-1, 1), xlabel="S from customs volumes (Chen, 2008)", ylabel="S from HGL values")
  ax.set_title(f"Spearman $\\rho$ = {rho:.2f} (n = {len(two):,} pair-years)", fontsize=8, loc="right")
  ax.legend(frameon=False, fontsize=7, loc="upper left", markerscale=2)
  panel_label(ax, "a")

  # Panel (b)
  ax = axes[1]
  shown = ban[ban["label"].isin(bans.loc[bans["exporter"].isin(panel_b_bans), "label"])]
  for (label, d), c in zip(shown.groupby("label", sort=False), [FOCUS, ACCENT, CONTRAST, GREY]):
    d = d[(d["years_since_ban"] >= window[0]) & (d["years_since_ban"] <= window[1]) & (d["I"] > 0)]
    ax.plot(d["years_since_ban"], d["share_declared"].clip(upper=1.5), marker="o", ms=3, lw=1.4, color=c, label=label)
  ax.axvline(0, color=GREY, lw=0.8, ls=":")
  ax.set(xlim=window, ylim=(0, 0.4), xlabel="Years since the export ban", ylabel="Exporter-declared / partner-recorded value")
  ax.legend(frameon=False, fontsize=7, loc="upper right")
  panel_label(ax, "b")

  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, two, ban #, cn

## Figure B.1: Reporting status of countries

Countries that switch at least once between reporting and appearing only through partners' data, sorted by total reconciled value.

In [ ]:
def fig_b1_reporting_status(con, years=STUDY_YEARS, figsize=(9, 17), name="fig_b1_reporting_status", save=SAVE_OUTPUTS):
  # Assign status = 2 if country actively reports trade, 1 if only in the network through partners
  status = query(con, f"""
    WITH roles AS (
      SELECT year, exporter AS country, E > 0 AS rep FROM panel WHERE year BETWEEN {years[0]} AND {years[1]}
      UNION ALL SELECT year, importer, I > 0 FROM panel WHERE year BETWEEN {years[0]} AND {years[1]}
    )
    SELECT year, country, CASE WHEN bool_or(rep) THEN 2 ELSE 1 END AS status FROM roles GROUP BY year, country
  """)
  # Table of countries (rows) status (vals) across the years (cols)
  grid = status.pivot(index="country", columns="year", values="status").reindex(columns=range(years[0], years[1] + 1)).fillna(0)
  # Filter those that have both statuses 2 and 1 (i.e. the switchers)
  switchers = grid[((grid == 2).sum(axis=1) > 0) & ((grid == 1).sum(axis=1) > 0)]
  # Total F calculated for labeling and sorting
  total_F = query(con, f"""SELECT country, sum(F) AS F FROM (SELECT exporter AS country, F FROM panel UNION ALL
                       SELECT importer, F FROM panel) GROUP BY country""").set_index("country")["F"]
  switchers = switchers.loc[total_F.reindex(switchers.index).fillna(0).sort_values(ascending=False).index]
  cmap = mpl.colors.ListedColormap(["#EEEEEE", ACCENT, FOCUS])

  # Plot
  fig, ax = plt.subplots(figsize=figsize)
  ax.imshow(switchers.values, aspect="auto", cmap=cmap, vmin=0, vmax=2, interpolation="nearest")
  ax.set_yticks(range(len(switchers)), [f"{c}  ({usd(total_F.get(c, 0))})" for c in switchers.index], fontsize=6.5)
  ax.set_xticks(range(0, years[1] - years[0] + 1, 5), switchers.columns[::5])
  ax.xaxis.tick_top()
  handles = [mpl.patches.Patch(color=FOCUS, label="Reports forest-product trade"),
             mpl.patches.Patch(color=ACCENT, label="In the network only through partner data"),
             mpl.patches.Patch(color="#EEEEEE", label="Absent from the panel")]
  ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, 1.03))
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, switchers

## Figures B.2-B.4: `heading x year` heatmaps

`stat` is any column of `heading_year_stats()`: **D_p50** (median `D`), **S_p50** (median `S`) or **n_flows** (number of two-sided flows).

Use `log=True` for counts (i.e., for **n_flows**), and `center=0` for signed values (i.e., for median `S`).

In [ ]:
def heading_year_stats(con, min_flow=MIN_FLOW, years=STUDY_YEARS):
  # Calculate stats per year x heading combination
  return query(con, f"""
    SELECT hs4, year, count(*) AS n_flows, median(D) AS D_p50, median(S) AS S_p50, sum(G) / sum(E + I) AS D_w
    FROM panel WHERE {sample_filter(min_flow, years)} GROUP BY hs4, year
  """)


def fig_b_heading_year_heatmap(con, stat="D_p50", min_flow=MIN_FLOW, years=STUDY_YEARS, cmap="magma_r", vmin=0, vmax=0.8,
                                  center=None, log=False, colorbar_label="Median D", mark_years=(2002, 2007, 2012, 2017),
                                  figsize=(10, 13), name="fig_b_heading_year_heatmap", save=SAVE_OUTPUTS):
  data = heading_year_stats(con, min_flow, years)
  grid = data.pivot(index="hs4", columns="year", values=stat).sort_index().reindex(columns=range(years[0], years[1] + 1))
  if center is not None:
    norm = mpl.colors.TwoSlopeNorm(vcenter=center, vmin=vmin, vmax=vmax)
  elif log:
    norm = mpl.colors.LogNorm(vmin=vmin, vmax=vmax)
  else:
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

  # Plot heatmap
  fig, ax = plt.subplots(figsize=figsize)
  image = ax.imshow(grid.values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
  ax.set_yticks(range(len(grid)), [f"{h} {HS4_NAMES[h]}" for h in grid.index], fontsize=7)
  # Color labels by chapter
  for tick, h in zip(ax.get_yticklabels(), grid.index):
    tick.set_color(CHAPTER_COLORS[h[:2]])
  # H-line every time new chapter is reached
  for i in range(1, len(grid)):
    if grid.index[i][:2] != grid.index[i - 1][:2]:
      ax.axhline(i - 0.5, color="white", lw=1.5)
  # V-line every year in `mark_years` (e.g. to show HS revision years)
  for yr in mark_years:
    ax.axvline(yr - years[0] - 0.5, color="white", lw=0.6, ls=":")
  ax.set_xticks(range(0, years[1] - years[0] + 1, 5), grid.columns[::5])
  ax.set_xlabel("Year (dotted lines are HS revision years)")
  fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02, label=colorbar_label, extend="both" if center is not None else "max")
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, grid

## Figure B.5: Value-weighted exporter discrepancy over time
Calculated for the largest `top_n` exporters

In [ ]:
def fig_b5_country_year_exporter_D(con, min_flow=MIN_FLOW, top_n=40, years=STUDY_YEARS, vmax=0.5, figsize=(10, 9),
                            name="fig_b5_country_year_exporter_D", save=SAVE_OUTPUTS):
  # Extract top_n exportes as list
  top = query(con, f"""SELECT exporter FROM panel WHERE year BETWEEN {years[0]} AND {years[1]}
                   GROUP BY exporter ORDER BY sum(F) DESC LIMIT {top_n}""")["exporter"].tolist()
  # Filter panel for selected exporters and calculate D_w
  data = query(con, f"""
    SELECT exporter, year, sum(G) / sum(E + I) AS D_w FROM panel
    WHERE {sample_filter(min_flow, years)} AND exporter IN ({", ".join(f"'{c}'" for c in top)})
    GROUP BY exporter, year
  """)
  # Table of exporter (row) D_w's (val) across years (col)
  grid = data.pivot(index="exporter", columns="year", values="D_w").reindex(index=top, columns=range(years[0], years[1] + 1))
  cmap = plt.get_cmap("magma_r").copy()
  cmap.set_bad("#DDDDDD")
  fig, ax = plt.subplots(figsize=figsize)
  image = ax.imshow(np.ma.masked_invalid(grid.values), aspect="auto", cmap=cmap, vmin=0, vmax=vmax, interpolation="nearest")

  ax.set_yticks(range(len(grid)), grid.index, fontsize=7.5)
  ax.set_xticks(range(0, years[1] - years[0] + 1, 5), grid.columns[::5])
  ax.set_xlabel("Year")

  fig.colorbar(image, ax=ax, label="Value-weighted D of exports (grey = no two-sided flows)", extend="max")
  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, grid

## Figure B.6: Two-sided flows with a reconciled value of zero

In [ ]:
def fig_b6_zero_F(con, years=STUDY_YEARS, bins_per_decade=10, figsize=(11, 4), name="fig_b6_zero_F", save=SAVE_OUTPUTS):
  # Group countries with: F==0 as zero_F, rounded base-10 logarithm of the reporting sum as log10_EI
  dist = query(con, f"""
    SELECT (F = 0) AS zero_F, floor(log10(greatest(E_nominal + I_nominal, 1)) * {bins_per_decade}) / {bins_per_decade} AS log10_EI, count(*) AS n_flows
    FROM panel WHERE {sample_filter(0, years)} GROUP BY ALL ORDER BY log10_EI
  """)
  # Share of annual flows with F=0
  annual = query(con, f"SELECT year, avg((F = 0)::INT) AS share_zero FROM panel WHERE {sample_filter(0, years)} GROUP BY year ORDER BY year")

  fig, axes = plt.subplots(1, 2, figsize=figsize)
  # Plot (a)
  ax = axes[0]
  # For those with F == 0: plot 10^log10_EI versus number of flows
  d = dist[dist["zero_F"] == True]
  ax.step(10 ** d["log10_EI"], d["n_flows"] / 1e3, where="mid", color=CONTRAST, lw=1.8, label="F = 0")
  # For those with F > 0: plot 10^log10_EI versus number of flows
  d = dist[dist["zero_F"] == False]
  ax.step(10 ** d["log10_EI"], d["n_flows"] / 1e3, where="mid", color=FOCUS, lw=1.8, label="F > 0")
  ax.set_xscale("log")
  ax.xaxis.set_major_formatter(USD_FORMATTER)
  ax.set(xlabel="E + I (sum of both partner reports, nominal USD, log scale)", ylabel="Two-sided flows (thousands)")
  ax.legend()
  panel_label(ax, "a")

  # Plot (b)
  ax = axes[1]
  ax.plot(annual["year"], 100 * annual["share_zero"], color=CONTRAST, lw=2, marker="o", ms=3)
  ax.set(xlim=years, ylim=(0, None), xlabel="Year", ylabel="Two-sided flows with F = 0 (%)")
  panel_label(ax, "b")

  fig.tight_layout()
  if save:
    save_figure(fig, name)
  return fig, dist, annual

## Table C.1: Summary statistics for all 68 headings

In [ ]:
def tab_c1_heading_summary(con, groups=None, min_flow=MIN_FLOW, name="tab_c1_heading_summary", save=SAVE_OUTPUTS):
  data = heading_stats(con, min_flow)
  # Groupings based on given `groups` parameter
  data["group"] = data["hs4"].map(groups) if groups is not None else ""
  # Table rows:
  table = pd.DataFrame({
    "HS4": data["hs4"], "Heading": data["name"], "Grp.": data["group"].astype(str),
    "F (B USD)": (data["F_total"] / 1e9).map(lambda v: fmt(v, 1)),
    "Two-sided (%)": data["two_sided_share"].map(lambda v: fmt(v, 0, pct=True)),
    "Flows": data["n_flows"].map(lambda v: f"{v:,.0f}"),
    "Med. D": data["D_p50"].map(lambda v: fmt(v, 2)), "$D_w$": data["D_w"].map(lambda v: fmt(v, 2)),
    "Med. S": data["S_p50"].map(lambda v: fmt(v, 2)), "$S_w$": data["S_w"].map(lambda v: fmt(v, 2)),
    "G/F (%)": data["G_over_F"].map(lambda v: fmt(v, 0, pct=True)),
  })

  if save:
    write_table(table, name)
  return table, data

## Table C.2: Largest exporters and importers

In [ ]:
def tab_c2_top_countries(con, min_flow=MIN_FLOW, top_n=20, name="tab_c2_top_countries", save=SAVE_OUTPUTS):
  roles = country_role_stats(con, min_flow)
  parts = []
  # First run Exporters (column suffix _x) and then Importers (_m)
  for suffix, title in [("_x", "Exporters"), ("_m", "Importers")]:
    # Sort by exporting / importing F value
    d = roles.sort_values("F" + suffix, ascending=False).head(top_n)
    # Calculate statistics and add to table parts
    parts.append(pd.DataFrame({
      "Country": d.index, "F (B USD)": (d["F" + suffix] / 1e9).map(lambda v: fmt(v, 0)),
      "Flows": d["n_flows" + suffix].map(lambda v: f"{v:,.0f}"), "Partners": d["n_partners" + suffix].astype(int),
      "Median D": d["D_p50" + suffix].map(lambda v: fmt(v, 2)), "$D_w$": d["D_w" + suffix].map(lambda v: fmt(v, 3)),
      "$S_w$": d["S_w" + suffix].map(lambda v: fmt(v, 3)),
      "G per flow (k USD)": (d["G_per_flow" + suffix] / 1e3).map(lambda v: fmt(v, 0)),
    }))
  table = pd.concat(parts, ignore_index=True)

  if save:
    write_table(table, name)
  return table, roles

# Run

Produce every thesis figure and table. Runs all functions above in the order in which the items appear in the thesis. The product groups of Table 4.2 and Table C.1 are set to the `k = 3` groups of Figure 4.7

## All Outputs

In [ ]:
PRODUCT_GROUPS_K = 3
PRODUCT_GROUP_NAMES = {1: "Bulk products", 2: "Added-value products",
                       3: "Specialty items"}

def make_all_outputs(con, save=SAVE_OUTPUTS):
  results = {}
  results["tab_2_1"] = tab_2_1_non_reporting(con, save=save)
  results["fig_4_1"] = fig_4_1_reporting_cases(con, save=save)
  results["fig_4_2"] = fig_4_2_annual_D(con, save=save)
  results["fig_4_3"] = fig_4_3_annual_S_G(con, save=save)
  results["fig_4_4"] = fig_4_4_D_by_flow_size(con, save=save)
  results["tab_4_1"] = tab_4_1_min_flow_sensitivity(con, save=save)
  results["fig_4_5"] = fig_4_5_variance_decomposition(con, save=save)
  results["fig_4_6"] = fig_4_6_heading_profiles(con, save=save)
  results["fig_4_7"] = fig_4_7_heading_clusters(con, color_k=PRODUCT_GROUPS_K, group_names=PRODUCT_GROUP_NAMES, save=save)
  groups = results["fig_4_7"][1].set_index("hs4")["group"]
  results["tab_4_2"] = tab_4_2_product_groups(con, groups, group_names=PRODUCT_GROUP_NAMES, save=save)
  results["fig_4_8"] = fig_4_8_country_roles(con, save=save)
  results["fig_4_9"] = fig_4_9_wood_exporters(con, save=save)
  results["fig_4_10"] = fig_4_10_dyad_concentration(con, save=save)
  results["fig_4_11"] = fig_4_11_persistence(con, save=save)
  results["tab_4_3"] = tab_4_3_top_persistent(con, save=save)
  results["fig_4_12"] = fig_4_12_network_basics(con, save=save)
  results["tab_4_4"] = tab_4_4_network_summary(con, save=save)
  results["tab_4_5"] = tab_4_5_robustness(con, save=save)
  results["fig_5_1"] = fig_5_1_external_validity(con, save=save)
  results["fig_5_2"] = fig_5_2_external_validity_cases(con, save=save)

  #Appendix figures and tables
  results["fig_b1"] = fig_b1_reporting_status(con, save=save)
  results["fig_b2"] = fig_b_heading_year_heatmap(con, "D_p50", name="fig_b2_heading_year_median_D", save=save)
  results["fig_b3"] = fig_b_heading_year_heatmap(con, "S_p50", cmap="RdBu", vmin=-0.3, vmax=0.3, center=0,
    colorbar_label="Median S (red where importer reports more, blue otherwise)", name="fig_b3_heading_year_median_S", save=save)
  results["fig_b4"] = fig_b_heading_year_heatmap(con, "n_flows", cmap="cividis", vmin=10, vmax=1e4, log=True,
    colorbar_label="Two-sided flows in the analysis sample (log scale)", name="fig_b4_heading_year_n", save=save)
  results["fig_b5"] = fig_b5_country_year_exporter_D(con, save=save)
  results["fig_b6"] = fig_b6_zero_F(con, save=save)
  results["tab_c1"] = tab_c1_heading_summary(con, groups, save=save)
  results["tab_c2"] = tab_c2_top_countries(con, save=save)
  plt.close("all")
  return results

## Automatically retrieve / build panel

In [ ]:
def get_panel(run_progress=None):
  """
  run_progress  : defaults to the automatically calculated RUN_PROGRESS in `LEGEND`
    1 = panel already exists, so simply connect to it and return the DuckDB connection
    2 = merged dataset exists, so enrich it, filter products, sanity check, build panel and connect to it
    3 = only source files given, so first merge, then create panel, then connect to it

    NOTE: this was not added as parameter value since overriding the automatically calculated default from `LEGEND`
          may overwrite previously created datasets (e.g. the merged or panel datasets). Change RUN_PROGRESS with care.
  """
  global RUN_PROGRESS

  # Feedback given unless run_progress is specified
  if run_progress is None:
    run_progress = RUN_PROGRESS
    if run_progress == 1:
      print(f"HS4 panel was found in [{PANEL_PATH}] -> Skipping to analysis")
    elif run_progress == 2:
      print(f"Merged database was found in [{MERGED_DB_PATH}] -> Skipping to panel creation")
      choice = input(f"Continue panel creation on path [{PANEL_PATH}]? (y/n): ").lower()
      if choice not in ("y", "yes", "1"):
          sys.exit(0)
    else:
      print(f"Directory with raw csv files to be merged exists as ./{SOURCE_DB_DIR}/")
      choice = input(f"Proceed with merging source files to [{MERGED_DB_PATH}]? (y/n): ").lower()
      if choice not in ("y", "yes", "1"):
          sys.exit(0)

  # DuckDB connection for pre-processing (if preprocessing is needed, i.e. run_progress != 1)
  con = duckdb.connect() if run_progress != 1 else None
  unchanged = run_progress == RUN_PROGRESS

  if run_progress == 3:
    merge_source_dataset(con)
    run_progress = 2
    RUN_PROGRESS = run_progress
    print("Source data files have been merged.\nNote: this step does NOT need to be performed again in future runs.")
    choice = input(f"Proceed with panel creation from merged DB [{MERGED_DB_PATH}] to panel [{PANEL_PATH}]? (y/n):").lower()
    if choice not in ("y", "yes", "1"):
        sys.exit(0)

  if run_progress == 2:
    enrich_raw_database(con)
    forest_table(con)
    check_duplicates(con)
    build_panel(con)
    run_progress = 1
    RUN_PROGRESS = run_progress
    print("Analysis panel created from merged data base.\nNote: this step does NOT need to be performed again in future runs.")
    con.close()

  panel_con = connect_panel()
  print("\nPanel Summary:")
  hs4_panel_summary(panel_con)
  return panel_con


## RUN - All

Run all figures and tables (commented out to avoid acccidentally overwriting results)

In [ ]:
con = get_panel()
results = make_all_outputs(con) # Set param `save` to False to avoid (over)writing the files to directory

## RUN - One

Run a single figure or table

In [ ]:
# con = get_panel()
# fig_4_6_heading_profiles(con, save=False)[0].show()
# display(tab_4_1_min_flow_sensitivity(con))